# 🎞️ Documental furry · INTRO hecha por el editor IA (sin personas, pura naturaleza)

**Qué hace:** el editor IA revisa **todas** tus tomas de cámara DJI y dron (nitidez, luz, movimiento y qué se ve con CLIP), escribe la narración leyendo tus documentos de *Guionistas y creatividad en papel*, la narra con **voz IA** y arma la intro con:

| Animaciones | Transiciones de tu guion |
|---|---|
| ✏️ **Real → dibujo de grafito** (lápiz, sombreado, papel) | **Desgarre** (los bordes se lijan como papel) |
| 🎨 **Pintura animada** estilo Veritasium (pinceladas + círculos, flechas y subrayados pintados) | **Enfoque → puntos de grafito** (rack focus) |
| 📅 **Tipografía por época**: 80s cromo, 90s VHS, 2000s máquina de escribir, 2010s noticiero, 2017 glitch, hoy limpio | **Túnel / obsesión** (entrar por la mirada) |
| 📼 **Calidad de imagen por época** (VHS 4:3, cámara digital 2001, HD 2014…) | **Memoria corrupta** (se congela + audio distorsionado 2.5 s) |
| 📰 **Escena de periódico** (gira, cae y se acerca al titular · recreación) | **Corte en acción** (portazo) · **Luz alta** (quemado a blanco) |

**🎬 Créditos impresos en el piso:** escribe tus créditos en la celda 2 (`CREDITOS`, separados por `|`) o sube `creditos.txt` a la carpeta del proyecto (uno por línea). Aparecen pintados en el suelo de las tomas, en perspectiva y siguiendo el movimiento de la cámara.

**Sonido:** música de documental generada (o la tuya en `Sonidos/Musica`), efectos (lápiz, pincel, papel, VHS, máquina de escribir, portazo… o los tuyos en `Sonidos/SFX sonidos`), sonido real de la naturaleza y ambiente de `Sonidos remasterizados (ambiente)`.

**Salida:** YouTube 16:9 (3-5 min) y TikTok 9:16 (60-90 s) en `Editores - clase 1/Editados/`, con la hoja de vista previa, el guion y las notas del editor.

**Cómo usarlo:** Entorno de ejecución → **H100** → **Ejecutar todo**. La celda 4 te pedirá permiso de Google para leer tus Google Docs.

In [ ]:
#@title 1 · Instalar herramientas (CLIP, voz IA, vLLM) y montar Google Drive
#@markdown Usa **Entorno de ejecución → Cambiar tipo → GPU H100**. La 1.ª vez tarda ~4-6 min.
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=False)          # fonemas en español para la voz Kokoro
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'kokoro>=0.9.4', 'soundfile',
                    'edge-tts', 'num2words', 'anthropic>=0.70', 'openai', 'google-api-python-client'], check=True)
    # vLLM en un entorno aparte para no romper el torch de Colab (lo usan CLIP y la voz)
    if not os.path.exists('/content/vllm_env/bin/vllm'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/vllm_env'], check=True)
        subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/vllm_env/bin/python', '-U', 'vllm'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'Sin GPU (irá más lento)')
except FileNotFoundError:
    print('Sin GPU (irá más lento)')

In [ ]:
#@title 2 · Configuración del documental (intro furry)
import os, glob, json, math, random, re, shutil, subprocess, base64, gc, time, unicodedata
import numpy as np, cv2
random.seed(7); np.random.seed(7)

#@markdown **Carpeta del proyecto en tu Drive** (la que tiene todas las subcarpetas organizadas).
PROYECTO = '/content/drive/MyDrive/Anuncio'  #@param {type:"string"}
#@markdown Carpetas con las tomas de naturaleza (separadas por coma). La IA revisa todo y elige lo mejor.
CARPETAS_VIDEO = "archivos-crudos-desordenados, Archivos dron desordenado"  #@param {type:"string"}
INCLUIR_IPHONE = False  #@param {type:"boolean"}
CARPETA_IPHONE = "Editores-clase 2/Camara-iphone-blopers"  #@param {type:"string"}
CARPETA_GUIONES = "Guionistas y creatividad en papel"  #@param {type:"string"}
CARPETA_SONIDOS = "Sonidos"  #@param {type:"string"}
#@markdown Fuentes propias (opcional): ponlas en esta carpeta y se usan con prioridad para esa época (nombre con el año, ej. `1998_mifuente.ttf`).
CARPETA_FUENTES = "Artistas/fuentes"  #@param {type:"string"}
CARPETA_SALIDA = "Editores - clase 1/Editados"  #@param {type:"string"}
TITULO_DOC = "FURRY: LA HISTORIA NO NARRADA"  #@param {type:"string"}

#@markdown ---
#@markdown **🎬 Créditos impresos en el piso** (como en las películas: aparecen "pintados" en el suelo de las tomas, en perspectiva y siguiendo el movimiento de la cámara).
#@markdown Escríbelos separados por `|`, ej.: `UN DOCUMENTAL DE ... | GUION: ... | EDICIÓN: ... | ANIMACIÓN: ...`
#@markdown (o pon un archivo `creditos.txt` en la carpeta del proyecto, uno por línea; si existe, manda ese).
CREDITOS = "GUSTAVO · DIRECCIÓN DE CÁMARAS | CLAUDE · DISEÑO Y EDICIÓN"  #@param {type:"string"}
#@markdown Crédito del final (sale en el piso debajo del título, al cierre).
CREDITO_FINAL = "UNA PRODUCCIÓN DE THE REPUBLIC OF FURRYS (BETA)"  #@param {type:"string"}
CREDITOS_DONDE = "al inicio"  #@param ["al inicio", "repartidos"]
#@markdown Fuente para los créditos: nombre de un archivo de tu carpeta de fuentes (vacío = Cinzel).
FUENTE_CREDITOS = "GUSTAVO · DIRECCIÓN DE CÁMARAS | CLAUDE · DISEÑO Y EDICIÓN"  #@param {type:"string"}
#@markdown Crédito del final (sale en el piso debajo del título, al cierre).
CREDITO_FINAL = "UNA PRODUCCIÓN DE THE REPUBLIC OF FURRYS (BETA)"  #@param {type:"string"}
CREDITOS_EN_TIKTOK = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Formatos**
HACER_YOUTUBE = True  #@param {type:"boolean"}
DURACION_YOUTUBE_MIN = 4.0  #@param {type:"slider", min:3, max:5, step:0.5}
HACER_TIKTOK = True  #@param {type:"boolean"}
DURACION_TIKTOK_S = 75  #@param {type:"slider", min:60, max:90, step:5}
SUBTITULOS_TIKTOK = True  #@param {type:"boolean"}
SUBTITULOS_YOUTUBE = False  #@param {type:"boolean"}
BARRAS_CINE_YOUTUBE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Guion y narración.** `ia` = la IA escribe la narración leyendo tus documentos (si falla, usa el guion incluido).
GUION = "ia"  #@param ["ia", "incluido"]
MOTOR_IA = "local_qwen"  #@param ["local_qwen", "claude", "simple"]
MODELO_LOCAL = "Qwen/Qwen3.8-27B"  #@param {type:"string"}
VOZ_IA = "kokoro"  #@param ["kokoro", "edge", "ninguna"]
VOZ_KOKORO = "em_alex"  #@param ["em_alex", "em_santa", "ef_dora"]
VOZ_EDGE = "es-MX-JorgeNeural"  #@param {type:"string"}
VELOCIDAD_VOZ = 0.92  #@param {type:"slider", min:0.8, max:1.1, step:0.02}

#@markdown ---
#@markdown **Imagen y sonido**
CAMARA_LENTA_NATURAL = True  #@param {type:"boolean"}
COLOR_AUTOMATICO = True  #@param {type:"boolean"}
SONIDO_NATURAL = True  #@param {type:"boolean"}
MUSICA = "generada"  #@param ["generada", "carpeta Sonidos/Musica", "sin musica"]

FPS = 24                                          # look de cine / documental
IN_COLAB = 'google.colab' in __import__('sys').modules
WORK = '/content/work_doc' if IN_COLAB else os.path.abspath('work_doc')
for d_ in ('shots', 'voz', 'sfx', 'fonts', 'audio'): os.makedirs(os.path.join(WORK, d_), exist_ok=True)
if not os.path.isdir(PROYECTO): raise FileNotFoundError(f"No encuentro {PROYECTO}. ¿Montaste Drive y el nombre es correcto?")
OUT_DIR = os.path.join(PROYECTO, CARPETA_SALIDA); os.makedirs(OUT_DIR, exist_ok=True)
CARPETAS = [c.strip() for c in CARPETAS_VIDEO.split(',') if c.strip()] + ([CARPETA_IPHONE] if INCLUIR_IPHONE else [])
FORMATOS = []
if HACER_YOUTUBE: FORMATOS.append({'name': 'youtube', 'W': 1920, 'H': 1080, 'target': DURACION_YOUTUBE_MIN * 60, 'subs': SUBTITULOS_YOUTUBE, 'guion': 'largo'})
if HACER_TIKTOK:  FORMATOS.append({'name': 'tiktok', 'W': 1080, 'H': 1920, 'target': float(DURACION_TIKTOK_S), 'subs': SUBTITULOS_TIKTOK, 'guion': 'corto'})
print("Proyecto:", PROYECTO); print("Material:", CARPETAS); print("Salida:", OUT_DIR); print("Formatos:", [f['name'] for f in FORMATOS])

In [ ]:
#@title 3 · El editor IA revisa TODO tu material: nitidez, luz, movimiento, y qué se ve en cada segundo (CLIP)
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))[:300]}\n{r.stderr[-1500:]}")
    return r

def probe(path):
    j = json.loads(run(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', path]).stdout)
    v = next((s for s in j['streams'] if s['codec_type'] == 'video'), None)
    a = next((s for s in j['streams'] if s['codec_type'] == 'audio'), None)
    if v is None: return None
    w, h = int(v['width']), int(v['height'])
    rot = 0
    for sd in v.get('side_data_list', []) or []:
        if 'rotation' in sd: rot = int(float(sd['rotation']))
    rot = int(v.get('tags', {}).get('rotate', rot) or rot)
    if abs(rot) % 180 == 90: w, h = h, w                      # videos verticales (celular / dron en vertical)
    num, den = (v.get('avg_frame_rate') or '30/1').split('/')
    fps = float(num) / max(1.0, float(den or 1))
    return {'duration': float(j['format'].get('duration', 0) or v.get('duration', 0) or 0), 'W': w, 'H': h, 'fps': fps,
            'trc': v.get('color_transfer'), 'audio': a is not None}

def pick_encoder():
    try:
        run(['ffmpeg', '-v', 'error', '-f', 'lavfi', '-i', 'color=s=256x256:d=0.2', '-c:v', 'h264_nvenc', '-f', 'null', '-'])
        return ['-c:v', 'h264_nvenc', '-preset', 'p5', '-cq', '18', '-b:v', '0']
    except Exception:
        return ['-c:v', 'libx264', '-preset', 'fast', '-crf', '17']
VENC = pick_encoder(); print("Encoder:", VENC[1])
HDR_TRC = {'arib-std-b67', 'smpte2084'}
def tonemap(info):
    """DJI en HLG / HDR → SDR Rec.709 (si no, se ve lavado)."""
    if info.get('trc') in HDR_TRC:
        return "zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p,"
    return ""

# ---------- Inventario (las .LRF de DJI son versiones chiquitas: sirven para analizar rápido) ----------
VID_EXT = ('.mp4', '.mov', '.m4v', '.mkv')
files, lrf = [], {}
for sub in CARPETAS:
    d_ = os.path.join(PROYECTO, sub)
    if not os.path.isdir(d_): print("⚠ No existe:", d_); continue
    for root, _, fs in os.walk(d_):
        for f in sorted(fs):
            p = os.path.join(root, f)
            if f.lower().endswith('.lrf'): lrf[os.path.splitext(f)[0].lower()] = p
            elif f.lower().endswith(VID_EXT) and os.path.getsize(p) > 0: files.append(p)
clips, seen = [], set()
for p in files:
    try: info = probe(p)
    except RuntimeError as e: print("  no se pudo leer", os.path.basename(p)); continue
    if info is None or info['duration'] < 2.0: continue
    key = (os.path.getsize(p), round(info['duration'], 1))
    if key in seen: print("  duplicado, omitido:", os.path.basename(p)); continue
    seen.add(key)
    c = {'src': p, 'name': os.path.basename(p), **info}
    small = lrf.get(os.path.splitext(c['name'])[0].lower())
    c['ana'] = small if small else p
    c['dron'] = 'dji_fly' in c['name'].lower() or 'dron' in p.lower()
    clips.append(c)
print(f"{len(clips)} tomas, {sum(c['duration'] for c in clips) / 60:.1f} min de material "
      f"({sum(c['dron'] for c in clips)} de dron, {sum(c['ana'] != c['src'] for c in clips)} con .LRF para análisis rápido)")
if not clips: raise RuntimeError("No encontré videos. Revisa CARPETAS_VIDEO.")

# ---------- CLIP: entiende qué hay en cada segundo ----------
try:
    import torch
    from transformers import CLIPModel, CLIPProcessor
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
    CLIP_M = CLIPModel.from_pretrained('openai/clip-vit-large-patch14', torch_dtype=torch.float16 if DEV == 'cuda' else torch.float32).to(DEV).eval()
    CLIP_P = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
    print("CLIP listo en", DEV)
except Exception as e:
    CLIP_M = None; print("⚠ CLIP no disponible (uso solo calidad de imagen):", str(e)[:200])
def clip_img(frames_bgr):
    with torch.no_grad():
        x = CLIP_P(images=[cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames_bgr], return_tensors='pt')['pixel_values'].to(DEV, CLIP_M.dtype)
        e = CLIP_M.get_image_features(pixel_values=x).float()
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()
def clip_txt(texts):
    with torch.no_grad():
        x = CLIP_P(text=list(texts), return_tensors='pt', padding=True, truncation=True).to(DEV)
        e = CLIP_M.get_text_features(**x).float()
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

ETIQUETAS = {'bosque': 'a forest with trees', 'río o arroyo': 'a river or stream', 'lago o mar': 'a lake or the sea',
             'montañas': 'mountains landscape', 'cielo y nubes': 'sky with clouds', 'atardecer': 'a sunset or golden hour sky',
             'niebla': 'fog and mist', 'camino': 'an empty road or path', 'flores y plantas': 'close-up of flowers and plants',
             'vista aérea': 'aerial drone view from above', 'animales': 'an animal in nature', 'ciudad': 'a city with buildings',
             'personas': 'a photo of a person or people'}
if CLIP_M is not None:
    LAB_E = clip_txt(list(ETIQUETAS.values()))
    PERSON_E = clip_txt(['a photo of a person', 'a selfie of a face', 'people standing'])
    NATURE_E = clip_txt(['a cinematic shot of nature', 'a landscape', 'plants and trees'])

def sample_frames(c, step):
    src = c['ana']; info = c if src == c['src'] else (probe(src) or c)
    w = 448; h = int(round(w * info['H'] / info['W'] / 2)) * 2
    vf = tonemap(info) + f"fps=1/{step},scale={w}:{h}:flags=area,format=bgr24"
    pr = subprocess.Popen(['ffmpeg', '-v', 'error', '-i', src, '-vf', vf, '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-'],
                          stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    out = []
    while True:
        b = pr.stdout.read(w * h * 3)
        if len(b) < w * h * 3: break
        out.append(np.frombuffer(b, np.uint8).reshape(h, w, 3))
    pr.wait(); return out

def colorfulness(f):
    b, g, r = [x.astype(np.float32) for x in cv2.split(f)]
    rg, yb = r - g, 0.5 * (r + g) - b
    return float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))

t0_ = time.time()
for i, c in enumerate(clips):
    step = 1.0 if c['duration'] < 180 else 2.0
    fr = sample_frames(c, step)
    if not fr: c['t'] = np.zeros(0); continue
    c['step'] = step; c['t'] = np.arange(len(fr)) * step + step / 2
    gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in fr]
    c['sharp'] = np.array([math.log(cv2.Laplacian(g_, cv2.CV_64F).var() + 1) for g_ in gray])
    c['luma'] = np.array([g_.mean() / 255 for g_ in gray])
    c['clipped'] = np.array([(g_ > 250).mean() + (g_ < 6).mean() for g_ in gray])
    c['color'] = np.array([colorfulness(f) for f in fr])
    c['sat'] = np.array([cv2.cvtColor(f, cv2.COLOR_BGR2HSV)[..., 1].mean() / 255 for f in fr])
    c['blue'] = np.array([(f[..., 0].astype(int) - f[..., 2]).mean() / 255 for f in fr])
    mot = [0.0] + [float(cv2.absdiff(gray[k], gray[k - 1]).mean()) for k in range(1, len(gray))]
    c['motion'] = np.array(mot); c['motion'][0] = c['motion'][1] if len(mot) > 1 else 0
    c['thumb'] = fr[len(fr) // 2]
    if CLIP_M is not None:
        embs = np.concatenate([clip_img(fr[k:k + 32]) for k in range(0, len(fr), 32)])
        c['emb'] = embs
        c['person'] = (embs @ PERSON_E.T).max(1) - (embs @ NATURE_E.T).max(1)
        lab = (embs @ LAB_E.T).mean(0); c['labels'] = [list(ETIQUETAS)[k] for k in np.argsort(-lab)[:2]]
    else:
        c['emb'] = None; c['person'] = np.zeros(len(fr)); c['labels'] = ['vista aérea'] if c['dron'] else ['naturaleza']
    print(f"\r  analizado {i + 1}/{len(clips)} ({time.time() - t0_:.0f}s)", end='')
print()
clips = [c for c in clips if len(c['t'])]
# Calidad comparada entre todas las tomas (z-scores)
def _z(key):
    allv = np.concatenate([c[key] for c in clips]); mu, sd = allv.mean(), allv.std() + 1e-6
    return mu, sd
(ms, ss), (mc, sc), (mm, sm) = _z('sharp'), _z('color'), _z('motion')
for c in clips:
    zm = (c['motion'] - mm) / sm
    c['q'] = (0.45 * (c['sharp'] - ms) / ss + 0.25 * (c['color'] - mc) / sc - 2.5 * c['clipped']
              - 1.5 * np.abs(c['luma'] - 0.45) - 0.5 * np.clip(zm - 1.5, 0, None)                 # sacudidas / movimientos bruscos
              - 3.0 * np.clip(c['person'] + 0.01, 0, None) * 10)                                    # nada de gente: pura naturaleza
    c['best'] = float(np.percentile(c['q'], 80))
ranking = sorted(clips, key=lambda c: -c['best'])
print("\n🎥 Las mejores tomas según el editor IA:")
for c in ranking[:12]:
    print(f"   {c['best']:+.2f}  {c['name'][:48]:48s} {c['duration']:5.0f}s  {c['fps']:.0f}fps  {', '.join(c['labels'])}")
n_people = sum(1 for c in clips if np.mean(c['person'] > 0) > 0.5)
if n_people: print(f"   ({n_people} tomas con personas quedan castigadas: el documental es pura naturaleza)")

In [ ]:
#@title 4 · Guion y narración: la IA lee tus documentos (Idea 1, convenciones, etc.) y escribe la intro en dos versiones
# tratamiento: normal | grafito (real → dibujo a lápiz) | pintura (pintura animada) | anio (tarjeta de época) | periodico | titulo
# transicion (al terminar el segmento): desgarre | enfoque_grafito | tunel | memoria_corrupta | corte_accion | luz_alta | disolver | corte
def S(texto, visual, tratamiento='normal', transicion='disolver', anio=0, lugar='', texto_pantalla='', titular='', anotacion='ninguna'):
    return dict(texto=texto, visual=visual, tratamiento=tratamiento, transicion=transicion, anio=anio, lugar=lugar,
                texto_pantalla=texto_pantalla, titular=titular, anotacion=anotacion)
GUION_LARGO = [
    S("Hay palabras que llegan antes que las personas. Entran primero a la habitación, se sientan… y cuando tú llegas, ya decidieron quién eres.",
      "thick fog in a dark forest at dawn", 'normal', 'tunel', texto_pantalla="ANTES DE CONOCERTE…"),
    S("Una de esas palabras es: furry.", "close-up of leaves and branches moving in the wind", 'titulo', 'corte_accion', texto_pantalla="FURRY"),
    S("Un furry es, simplemente, alguien a quien le apasionan los personajes animales con rasgos humanos. Los que dibuja, los que escribe, "
      "los que crea como una versión de sí mismo: su fursona. Es arte, es comunidad… y para muchísimas personas, es un refugio.",
      "birds flying over a calm lake", 'pintura', 'desgarre', texto_pantalla="FURSONA", anotacion='circulo'),
    S("Pero afuera de esa comunidad, la historia que se contó fue otra. Una historia escrita por titulares.",
      "an empty road through the forest", 'grafito', 'enfoque_grafito', texto_pantalla="UNA HISTORIA ESCRITA POR TITULARES"),
    S("Todo empezó en 1989, en Costa Mesa, California. ConFurence cero: unas decenas de personas en un hotel. La primera convención dedicada solo a esto.",
      "golden hour sunlight through trees", 'anio', 'disolver', anio=1989, lugar="Costa Mesa, California", texto_pantalla="CONFURENCE 0"),
    S("En 1998, un periodista británico entró a ConFurence nueve. Escribía para una revista masculina llamada Loaded. Su reportaje se tituló "
      "Heavy Petting… y en unas cuantas páginas, convirtió a toda una comunidad en un escándalo.",
      "dark clouds over mountains", 'periodico', 'memoria_corrupta', anio=1998, lugar="California, EE. UU.",
      titular="«HEAVY PETTING»: LA CONVENCIÓN QUE UNA REVISTA CONVIRTIÓ EN ESCÁNDALO"),
    S("La reacción llegó desde adentro: un grupo llamado Burned Furs exigía «limpiar» el fandom. Fue la primera grieta. Y no sería la última.",
      "cracked dry ground or rocks", 'grafito', 'desgarre', anio=1998, texto_pantalla="BURNED FURS"),
    S("En marzo de 2001, Vanity Fair publicó Pleasures of the Fur. Los entrevistados denunciaron que sacaron sus palabras de contexto. "
      "Pero ya no importaba: el titular viajó más rápido que la verdad.",
      "wind moving tall grass", 'periodico', 'luz_alta', anio=2001, lugar="Estados Unidos",
      titular="«PLEASURES OF THE FUR»: EL REPORTAJE QUE MARCÓ UNA COMUNIDAD"),
    S("Las convenciones aprendieron a cerrar la puerta. Prensa con escolta, voceros, comunicados. El miedo a la cámara se volvió protocolo.",
      "a closed path between trees", 'pintura', 'corte_accion', anio=2001, texto_pantalla="PRENSA CON ESCOLTA", anotacion='subrayado'),
    S("Y entonces, el odio dejó de ser solo tinta. 7 de diciembre de 2014, Rosemont, Illinois. Madrugada en el Midwest FurFest. "
      "Alguien deja polvo de cloro en una escalera del hotel. Miles de personas evacuadas al frío. Diecinueve, al hospital.",
      "cold winter sky at night or dusk", 'anio', 'memoria_corrupta', anio=2014, lugar="Rosemont, Illinois", texto_pantalla="19 HOSPITALIZADOS"),
    S("El responsable… nunca fue encontrado.", "dense fog hiding the landscape", 'grafito', 'corte_accion', anio=2014, texto_pantalla="CASO SIN RESOLVER"),
    S("En 2017, en Denver, un grupo de extrema derecha llamado Furry Raiders, con brazaletes que imitaban la simbología nazi, partió al fandom en dos. "
      "Amenazas, protestas… y la Rocky Mountain Fur Con terminó cancelada para siempre.",
      "storm clouds moving fast over the land", 'periodico', 'tunel', anio=2017, lugar="Denver, Colorado",
      titular="CONVENCIÓN DE DENVER, CANCELADA TRAS AMENAZAS"),
    S("Hoy el ataque cambió de forma: memes, acoso, linchamientos digitales. Gente que nunca conoció a un furry… pero que ya sabe exactamente qué odiar.",
      "aerial view of a lonely tree in a field", 'grafito', 'enfoque_grafito', anio=2024, texto_pantalla="GENTE ANTIFURRY", anotacion='flecha'),
    S("Esta es la historia que nadie te contó completa. La de una comunidad juzgada por sus titulares… y la de lo que pasó cuando el miedo, de verdad, tocó la puerta.",
      "epic aerial shot of mountains and valley at sunset", 'titulo', 'luz_alta', texto_pantalla=TITULO_DOC),
]
GUION_CORTO = [
    S("Esta palabra llegó antes que las personas. Y cuando ellas llegaron, ya las habían juzgado: furry.",
      "thick fog in a dark forest at dawn", 'titulo', 'corte_accion', texto_pantalla="FURRY"),
    S("Un furry es alguien que ama a los personajes animales con rasgos humanos: los dibuja, los escribe… y crea el suyo, su fursona.",
      "birds flying over a calm lake", 'pintura', 'desgarre', texto_pantalla="FURSONA", anotacion='circulo'),
    S("Pero en 1998, una revista británica entró a una de sus convenciones… y en unas páginas, los convirtió en un escándalo.",
      "dark clouds over mountains", 'periodico', 'memoria_corrupta', anio=1998, lugar="California, EE. UU.",
      titular="«HEAVY PETTING»: LA CONVENCIÓN QUE UNA REVISTA CONVIRTIÓ EN ESCÁNDALO"),
    S("En 2001, Vanity Fair repitió la fórmula. Los entrevistados dijeron que sacaron sus palabras de contexto. El titular corrió más rápido que la verdad.",
      "wind moving tall grass", 'grafito', 'enfoque_grafito', anio=2001, texto_pantalla="FUERA DE CONTEXTO"),
    S("Y en 2014, el odio dejó de ser tinta: alguien dejó cloro en una convención. Diecinueve personas al hospital. Nunca encontraron al culpable.",
      "cold winter sky at dusk", 'anio', 'corte_accion', anio=2014, lugar="Rosemont, Illinois", texto_pantalla="CASO SIN RESOLVER"),
    S("Esta es la historia que nadie te contó completa.", "epic aerial shot of mountains at sunset", 'titulo', 'luz_alta', texto_pantalla=TITULO_DOC),
]

# ---------- Leer tus documentos de guion (Google Docs vía API de Drive, o .txt/.md/.docx) ----------
DOCS = {}
gdir = os.path.join(PROYECTO, CARPETA_GUIONES)
if os.path.isdir(gdir):
    for f in sorted(os.listdir(gdir)):
        p = os.path.join(gdir, f)
        if f.lower().endswith(('.txt', '.md')): DOCS[f] = open(p, encoding='utf-8', errors='ignore').read()
        elif f.lower().endswith('.docx'):
            try:
                import zipfile; x = zipfile.ZipFile(p).read('word/document.xml').decode('utf8', 'ignore')
                DOCS[f] = re.sub(r'<[^>]+>', ' ', x.replace('</w:p>', '\n'))
            except Exception: pass
if IN_COLAB and GUION == 'ia':
    try:                                                      # los Google Docs en Drive son "atajos": se exportan como texto con la API
        from google.colab import auth; auth.authenticate_user()
        from googleapiclient.discovery import build
        svc = build('drive', 'v3')
        q = f"name = '{CARPETA_GUIONES}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
        for fo in svc.files().list(q=q, fields='files(id)').execute().get('files', []):
            for d_ in svc.files().list(q=f"'{fo['id']}' in parents and mimeType = 'application/vnd.google-apps.document' and trashed = false",
                                      fields='files(id,name)').execute().get('files', []):
                if d_['name'].lower().startswith('copia de'): continue
                DOCS[d_['name']] = svc.files().export(fileId=d_['id'], mimeType='text/plain').execute().decode('utf-8', 'ignore')
    except Exception as e:
        print("⚠ No pude leer los Google Docs (uso los datos incluidos):", str(e)[:200])
print("Documentos leídos:", list(DOCS) or '— (uso el guion incluido)')

TRAT = ['normal', 'grafito', 'pintura', 'anio', 'periodico', 'titulo']
TRANS_K = ['desgarre', 'enfoque_grafito', 'tunel', 'memoria_corrupta', 'corte_accion', 'luz_alta', 'disolver', 'corte']
STR = {'type': 'string'}
SEG_SCHEMA = {'type': 'object', 'additionalProperties': False,
              'required': ['texto', 'visual', 'tratamiento', 'transicion', 'anio', 'lugar', 'texto_pantalla', 'titular', 'anotacion'],
              'properties': {'texto': STR, 'visual': STR, 'tratamiento': {'type': 'string', 'enum': TRAT},
                             'transicion': {'type': 'string', 'enum': TRANS_K}, 'anio': {'type': 'integer'}, 'lugar': STR,
                             'texto_pantalla': STR, 'titular': STR, 'anotacion': {'type': 'string', 'enum': ['ninguna', 'circulo', 'flecha', 'subrayado']}}}
SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['largo', 'corto'],
          'properties': {'largo': {'type': 'array', 'items': SEG_SCHEMA}, 'corto': {'type': 'array', 'items': SEG_SCHEMA}}}
W_LARGO = (int(DURACION_YOUTUBE_MIN * 60 * 1.75), int(DURACION_YOUTUBE_MIN * 60 * 2.3))     # palabras (~2.6 palabras/s con pausas)
W_CORTO = (int(DURACION_TIKTOK_S * 1.5), int(DURACION_TIKTOK_S * 2.1))

# ---------- Motor de IA (mismo que en los otros cuadernos) ----------
VLLM = {'proc': None, 'model': None}
MODELO_RESPALDO = "Qwen/Qwen3-VL-8B-Instruct"
def _serve(model, util, ctx):
    import urllib.request
    vllm_bin = '/content/vllm_env/bin/vllm'
    if not os.path.exists(vllm_bin): raise RuntimeError("vLLM no está instalado (corre la celda 1 en Colab).")
    log_path = f"{WORK}/vllm.log"; log = open(log_path, 'w')
    proc = subprocess.Popen([vllm_bin, 'serve', model, '--port', '8000', '--max-model-len', str(ctx),
                             '--gpu-memory-utilization', str(util), '--limit-mm-per-prompt', json.dumps({'image': 16}),
                             '--served-model-name', 'local'], stdout=log, stderr=subprocess.STDOUT)
    print(f"Cargando {model} en la GPU (la 1.ª vez lo descarga, 5-15 min)…"); t0 = time.time()
    while True:
        try: urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); break
        except Exception:
            if proc.poll() is not None:
                shutil.copy(log_path, os.path.join(OUT_DIR, 'vllm.log'))
                raise RuntimeError(f"vLLM se detuvo con {model}:\n{open(log_path).read()[-2500:]}")
            if time.time() - t0 > 40 * 60: proc.terminate(); raise RuntimeError("vLLM tardó demasiado")
            time.sleep(10)
    print(f"Modelo listo en {(time.time() - t0) / 60:.1f} min"); return proc
def llm_start():
    if VLLM['proc'] is not None and VLLM['proc'].poll() is None: return
    errors = []
    for model, util, ctx in ((MODELO_LOCAL, 0.80, 32768), (MODELO_LOCAL, 0.85, 16384), (MODELO_RESPALDO, 0.5, 32768)):
        try: VLLM['proc'] = _serve(model, util, ctx); VLLM['model'] = model; return
        except RuntimeError as e:
            errors.append(str(e)[-600:]); print(f"⚠ No arrancó {model} (util {util}, ctx {ctx}); pruebo otra configuración…")
            subprocess.run(['pkill', '-f', 'vllm serve'], capture_output=True); time.sleep(5)
    raise RuntimeError("Ningún modelo arrancó:\n" + "\n---\n".join(errors))
def llm_stop():
    p_ = VLLM.get('proc')
    if p_ is not None and p_.poll() is None:
        p_.terminate()
        try: p_.wait(timeout=60)
        except Exception: p_.kill()
    VLLM['proc'] = None
def ask_llm(parts, schema, name='respuesta', max_tokens=16000):
    if MOTOR_IA == 'local_qwen':
        from openai import OpenAI
        llm_start()
        content = [{'type': 'text', 'text': v} if k == 'text' else {'type': 'image_url', 'image_url': {'url': f"data:image/jpeg;base64,{v}"}} for k, v in parts]
        r = OpenAI(base_url='http://localhost:8000/v1', api_key='local').chat.completions.create(
            model='local', messages=[{'role': 'user', 'content': content}], max_tokens=max_tokens, temperature=0.7, top_p=0.8,
            presence_penalty=1.0, response_format={'type': 'json_schema', 'json_schema': {'name': name, 'schema': schema}},
            extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}})
        txt = r.choices[0].message.content.strip(); return json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
    if MOTOR_IA == 'claude':
        try: from google.colab import userdata; key = userdata.get('ANTHROPIC_API_KEY')
        except Exception: key = os.environ.get('ANTHROPIC_API_KEY')
        if not key: raise RuntimeError("Falta el secreto ANTHROPIC_API_KEY")
        import anthropic
        content = [{'type': 'text', 'text': v} if k == 'text' else {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': v}} for k, v in parts]
        with anthropic.Anthropic(api_key=key).beta.messages.stream(
            model='claude-opus-5-5', max_tokens=max_tokens, betas=['server-side-fallback-2026-07-01'], fallbacks='default',
            output_config={'effort': 'high', 'format': {'type': 'json_schema', 'schema': schema}},
            messages=[{'role': 'user', 'content': content}]) as stream:
            msg = stream.get_final_message()
        if msg.stop_reason == 'refusal': raise RuntimeError("Claude declinó la petición")
        return json.loads(next(b_.text for b_ in msg.content if b_.type == 'text'))
    raise RuntimeError("MOTOR_IA = simple (sin IA)")

def check_guion(g):
    probs = []
    for key, (lo, hi), n_ in (('largo', W_LARGO, (8, 20)), ('corto', W_CORTO, (4, 9))):
        segs = g.get(key) or []
        w = sum(len(s['texto'].split()) for s in segs)
        if not (n_[0] <= len(segs) <= n_[1]): probs.append(f"'{key}' debe tener entre {n_[0]} y {n_[1]} segmentos (tiene {len(segs)})")
        if not (lo * 0.8 <= w <= hi * 1.15): probs.append(f"'{key}' debe tener entre {lo} y {hi} palabras de narración (tiene {w})")
        if segs and segs[-1]['tratamiento'] != 'titulo': probs.append(f"'{key}' debe terminar con tratamiento 'titulo'")
        for s in segs:
            if s['tratamiento'] in ('anio', 'periodico') and not (1980 <= s['anio'] <= 2030): probs.append(f"falta 'anio' en: {s['texto'][:40]}")
            if s['tratamiento'] == 'periodico' and not s['titular'].strip(): probs.append(f"falta 'titular' en: {s['texto'][:40]}")
    return probs

GUIONES = {'largo': GUION_LARGO, 'corto': GUION_CORTO}; GUION_FUENTE = 'incluido'
if GUION == 'ia' and MOTOR_IA != 'simple' and DOCS:
    docs_txt = "\n\n".join(f"=== {k} ===\n{v[:14000]}" for k, v in DOCS.items())
    footage = "\n".join(f"- {c['name']}: {', '.join(c['labels'])}{' (dron)' if c['dron'] else ''}" for c in ranking[:40])
    thumbs = []
    for c in ranking[:12]:
        ok, buf = cv2.imencode('.jpg', c['thumb'], [cv2.IMWRITE_JPEG_QUALITY, 75]); thumbs.append(('img', base64.standard_b64encode(buf.tobytes()).decode()))
    ejemplo = json.dumps(GUION_CORTO[:3], ensure_ascii=False)
    PROMPT = f"""Eres guionista y editor de documentales (estilo YouTube de alta calidad: Veritasium, Vox, documentales de Netflix).
Escribe la NARRACIÓN en español latino para la INTRO de un documental sobre el furry fandom: «{TITULO_DOC}».
El video NO tiene personas: solo tomas de naturaleza sin voz (arriba van fotogramas del mejor material; lista de lo que hay:
{footage})
El guion original del equipo («Idea 1») pide: escena 1 = qué es un furry y cómo se difamó a la comunidad; luego la historia no narrada
del primer caso polémico del fandom con su AÑO y la tipografía y calidad de imagen de esa época; cambios de grabación real a dibujo de
grafito; escena 2 = gente antifurry, casos de odio; escena de periódico con la noticia; transiciones: desgarre (bordes que se desgastan
como papel lijado), enfoque que se vuelve puntos de grafito, túnel por la mirada (obsesión), memoria corrupta (la imagen se congela y el
audio se distorsiona), corte en acción (un portazo), desvanecimiento por luz alta.

REGLAS:
- Usa SOLO hechos que estén en los documentos de abajo (años, lugares, nombres de convenciones, cifras). No inventes datos ni citas.
- Cuando algo es una acusación o denuncia contra una persona, dilo como acusación ("fue acusado", "denunciaron"); mejor enfócate en
  las convenciones y la comunidad, no en ataques personales. Nada sexual explícito: es un video para YouTube/TikTok.
- Voz de narrador de documental: frases cortas, ritmo, pausas con «…», intriga; el gancho en la primera frase.
- 'largo' (YouTube, ~{DURACION_YOUTUBE_MIN:.1f} min): entre {W_LARGO[0]} y {W_LARGO[1]} palabras en total, 10-16 segmentos.
  'corto' (TikTok vertical, ~{DURACION_TIKTOK_S} s): entre {W_CORTO[0]} y {W_CORTO[1]} palabras, 5-7 segmentos, gancho fuerte en 2 s.
- Cada segmento: 'texto' (lo que dice la voz), 'visual' (en INGLÉS, qué toma de naturaleza va debajo, ej. "fog in a dark forest",
  que combine con el ánimo de esa frase y con el material disponible), 'tratamiento' ({', '.join(TRAT)}), 'transicion' a la
  siguiente ({', '.join(TRANS_K)}; varía, usa todas las del guion original al menos una vez en 'largo'), 'anio' (año del hecho o 0),
  'lugar' (ciudad/país o ""), 'texto_pantalla' (máx. 5 palabras en MAYÚSCULAS que aparecen animadas, o ""), 'titular' (titular de
  periódico en MAYÚSCULAS si tratamiento='periodico', si no ""), 'anotacion' (ninguna/circulo/flecha/subrayado: pintura animada
  sobre la imagen, úsala con 'pintura' y 'grafito').
- Usa 'anio' en los segmentos de cada caso histórico (así la tipografía y la calidad de imagen cambian según la época).
- El último segmento de cada versión es tratamiento 'titulo' con texto_pantalla = "{TITULO_DOC}" y transicion 'luz_alta'.
Ejemplo de formato (no lo copies): {ejemplo}

DOCUMENTOS DEL EQUIPO:
{docs_txt}"""
    parts = thumbs + [('text', PROMPT)]
    for intento in range(3):
        try:
            g = ask_llm(parts, SCHEMA, 'guion_documental', 12000)
            probs = check_guion(g)
            if not probs:
                GUIONES = {'largo': g['largo'], 'corto': g['corto']}; GUION_FUENTE = 'ia'; break
            print(f"⚠ Intento {intento + 1}: el guion no cumple → {'; '.join(probs[:4])}")
            parts = thumbs + [('text', PROMPT + "\n\nTu respuesta anterior tenía estos problemas, corrígelos:\n- " + "\n- ".join(probs)
                               + "\nRespuesta anterior:\n" + json.dumps(g, ensure_ascii=False)[:12000])]
        except Exception as e:
            print(f"⚠ Intento {intento + 1} falló:", str(e)[-400:])
    if GUION_FUENTE != 'ia': print("→ Uso el guion incluido (escrito a partir de tus documentos).")
llm_stop()
for k_, segs in GUIONES.items():
    for s in segs: s['anio'] = int(s.get('anio') or 0)
    w = sum(len(s['texto'].split()) for s in segs)
    print(f"\n📝 Guion {k_} ({GUION_FUENTE}, {len(segs)} segmentos, {w} palabras):")
    for i, s in enumerate(segs):
        tag = f"[{s['tratamiento']}{' ' + str(s['anio']) if s['anio'] else ''} → {s['transicion']}]"
        print(f"  {i + 1:2d}. {tag:34s} {s['texto'][:110]}")
with open(os.path.join(OUT_DIR, 'guion_documental.json'), 'w', encoding='utf-8') as f_:
    json.dump(GUIONES, f_, ensure_ascii=False, indent=1)

In [ ]:
#@title 5 · Voz IA del narrador (Kokoro en la GPU · o Edge · con respaldo) + tratamiento de voz de documental
SR = 48000
PRONUNCIA = {"Vanity Fair": "Vániti Fer", "Loaded": "Lóuded", "Heavy Petting": "Jévi Péting", "Pleasures of the Fur": "Plésures of de Fer",
             "Midwest FurFest": "Mídwest Fur Fest", "ConFurence": "Con Fíurens", "Burned Furs": "Bernd Furs", "Rosemont": "Róusmont",
             "Illinois": "Ilinói", "Furry Raiders": "Furri Réiders", "Rocky Mountain Fur Con": "Róqui Máuntein Fur Con",
             "Rainfurrest": "Rein fúrest", "Anthrocon": "Ántrocon", "fandom": "fándom", "furries": "furris", "furry": "furri",
             "Fur Affinity": "Fur Afíniti", "fursona": "fursona", "Denver": "Dénver"}
def texto_para_voz(t):
    try:
        from num2words import num2words
        t = re.sub(r'\b(\d{1,4})\b', lambda m: num2words(int(m.group(1)), lang='es'), t)
    except Exception: pass
    if VOZ_IA == 'kokoro':
        for a, b in sorted(PRONUNCIA.items(), key=lambda kv: -len(kv[0])): t = re.sub(re.escape(a), b, t, flags=re.I)
    return t.replace('«', '').replace('»', '').replace('…', '... ')

_KOKORO = {}
def tts_kokoro(text, out):
    import soundfile as sf
    if 'p' not in _KOKORO:
        from kokoro import KPipeline
        _KOKORO['p'] = KPipeline(lang_code='e', repo_id='hexgrad/Kokoro-82M')
    chunks = [(a.detach().cpu().numpy() if hasattr(a, 'detach') else np.asarray(a)) for _, _, a in _KOKORO['p'](text, voice=VOZ_KOKORO, speed=VELOCIDAD_VOZ, split_pattern=r'(?<=[.!?…])\s+')]
    pause = np.zeros(int(24000 * 0.28), np.float32)          # respiración entre frases
    audio = np.concatenate(sum([[np.asarray(a, np.float32), pause] for a in chunks], [])[:-1])
    sf.write(out + '.raw.wav', audio, 24000); return out + '.raw.wav'
def tts_edge(text, out):
    rate = f"{int(round((VELOCIDAD_VOZ - 1) * 100)):+d}%"
    run(['edge-tts', '--voice', VOZ_EDGE, '--rate', rate, '--text', text, '--write-media', out + '.raw.mp3']); return out + '.raw.mp3'
def tts_espeak(text, out):
    run(['espeak-ng', '-v', 'es-419', '-s', '150', '-w', out + '.raw.wav', text]); return out + '.raw.wav'

VOICE_FX = ("highpass=f=70,equalizer=f=170:t=q:w=1:g=2.5,equalizer=f=3200:t=q:w=1.4:g=1.5,"
            "acompressor=threshold=-20dB:ratio=3:attack=5:release=150:makeup=2,aecho=0.8:0.5:32:0.10,alimiter=limit=0.9")
def read_wav_mono(path):
    r = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', str(SR), '-'], capture_output=True)
    return np.frombuffer(r.stdout, np.float32).copy()
MOTORES_VOZ = {'kokoro': [tts_kokoro, tts_edge, tts_espeak], 'edge': [tts_edge, tts_kokoro, tts_espeak], 'ninguna': []}[VOZ_IA]
VOZ_USADA = set()
def synth(text, out):
    for fn in MOTORES_VOZ:
        try:
            raw = fn(texto_para_voz(text), out)
            run(['ffmpeg', '-y', '-v', 'error', '-i', raw, '-af', VOICE_FX, '-ac', '1', '-ar', str(SR), out])
            VOZ_USADA.add(fn.__name__[4:]); return read_wav_mono(out)
        except Exception as e:
            print(f"   ⚠ voz {fn.__name__[4:]} falló: {str(e)[-160:]}")
    VOZ_USADA.add('silencio')                                  # sin voz: deja el tiempo de lectura para los textos
    return np.zeros(int(SR * max(1.5, len(text.split()) / 2.6)), np.float32)

for key, segs in GUIONES.items():
    if not any(f['guion'] == key for f in FORMATOS): continue
    tot = 0
    for i, s in enumerate(segs):
        s['voz'] = synth(s['texto'], f"{WORK}/voz/{key}_{i:02d}.wav"); s['vd'] = len(s['voz']) / SR; tot += s['vd']
        print(f"\r  voz {key}: {i + 1}/{len(segs)}", end='')
    print(f"  → {tot:.0f} s de narración")
print("Motor de voz:", ', '.join(sorted(VOZ_USADA)))
if 'silencio' in VOZ_USADA: print("⚠ Algún segmento quedó sin voz (revisa los avisos de arriba).")

In [ ]:
#@title 6 · El editor IA arma la línea de tiempo y elige la mejor toma para cada frase (CLIP + calidad + sin repetir)
TR = {'desgarre': (0.5, 0.5, 0), 'enfoque_grafito': (0.9, 0.9, 0), 'tunel': (0.7, 0.7, 0), 'memoria_corrupta': (0, 0, 2.5),
      'corte_accion': (0, 0.2, 0), 'luz_alta': (0.7, 0.8, 0), 'disolver': (0.4, 0.4, 0), 'corte': (0, 0, 0)}
LEAD = {'normal': 0.6, 'grafito': 0.5, 'pintura': 1.4, 'anio': 1.9, 'periodico': 1.7, 'titulo': 1.0}
PAD = 1.2
def era_of(segs):
    """El año 'se arrastra': después de un caso de 1998, lo siguiente sigue en esa época hasta que aparezca otro año."""
    y = 0
    for s in segs:
        if s['anio']: y = s['anio']
        s['era'] = y if s['tratamiento'] not in ('titulo',) else 0

def build_timeline(fmt):
    segs = [dict(s) for s in GUIONES[fmt['guion']]]
    era_of(segs)
    for i, s in enumerate(segs):
        s['tratamiento'] = s['tratamiento'] if s['tratamiento'] in LEAD else 'normal'
        s['transicion'] = s['transicion'] if s['transicion'] in TR else 'disolver'
        if i == len(segs) - 1: s['transicion'] = 'luz_alta'
        s['lead'] = LEAD[s['tratamiento']] + (1.2 if i == 0 else 0)
        s['tail'] = 3.2 if i == len(segs) - 1 else 0.6
    def total(): return sum(s['lead'] + s['vd'] + s['tail'] + TR[s['transicion']][2] for s in segs[:-1]) + segs[-1]['lead'] + segs[-1]['vd'] + segs[-1]['tail']
    extra = fmt['target'] - total()
    if extra > 0:                                       # "respiros": tomas de pura naturaleza + música entre ideas
        slots = [s for s in segs[:-1] if s['transicion'] in ('desgarre', 'luz_alta', 'tunel', 'disolver', 'enfoque_grafito')
                 and s['tratamiento'] in ('normal', 'grafito', 'pintura', 'anio')] or segs[:-1]
        cap = 5.0 if fmt['H'] < fmt['W'] else 1.5
        for s in slots:
            add = min(cap, extra / max(1, len(slots))); s['tail'] += add
        segs[-1]['tail'] += min(4.0, max(0, fmt['target'] - total()))
    f = 0
    for i, s in enumerate(segs):
        s['D'] = int(round((s['lead'] + s['vd'] + s['tail']) * FPS)); s['f0'] = f; s['f1'] = f + s['D']
        s['voz_f'] = f + int(round(s['lead'] * FPS))
        pre, post, gap = TR[s['transicion']]
        s['pre'], s['post'], s['gap'] = int(pre * FPS), int(post * FPS), int(gap * FPS)
        f = s['f1'] + (s['gap'] if i < len(segs) - 1 else 0)
    fmt['segs'] = segs; fmt['L'] = f
    print(f"⏱ {fmt['name']}: {f / FPS:.0f} s ({f / FPS / 60:.1f} min) · objetivo {fmt['target']:.0f} s")
    if f / FPS > fmt['target'] * 1.2: print("   (la narración es más larga que el objetivo; puedes subir VELOCIDAD_VOZ o acortar el guion)")

GENERIC_Q = "beautiful cinematic nature footage"
if CLIP_M is not None: GEN_E = clip_txt([GENERIC_Q])[0]
KEYS = [(r'fog|mist|niebla', lambda c: -2 * c['sat']), (r'sunset|golden|amanecer|dawn|atardecer', lambda c: 3 * (-c['blue'])),
        (r'water|lake|river|sea|stream|lago|rio', lambda c: 3 * c['blue']), (r'night|dark|dusk|storm|cold', lambda c: -2 * c['luma']),
        (r'grass|leaves|plants|flowers|trees|forest', lambda c: 2 * c['sat'])]
def sim_curve(c, query, q_e):
    if c.get('emb') is not None and q_e is not None: return 10 * (c['emb'] @ q_e)
    v = np.full(len(c['t']), 0.6 if (c['dron'] and re.search(r'aerial|drone|above|valley', query)) else 0.0)
    for pat, fn in KEYS:
        if re.search(pat, query, re.I): v = v + fn(c)
    return v + np.random.uniform(0, 0.3)

def pick_shot(query, q_e, dur, used, prev_clip):
    best = None
    for ci, c in enumerate(clips):
        speed = 0.5 if CAMARA_LENTA_NATURAL and c['fps'] >= 47 else 1.0
        if (dur + 2 * PAD) * speed > c['duration'] - 0.2: speed = 1.0          # toma corta: sin cámara lenta
        need = (dur + 2 * PAD) * speed
        if need > c['duration'] - 0.2: continue
        sc = sim_curve(c, query, q_e) + 0.6 * c['q']
        st = c['step']; n = max(1, int(round(dur * speed / st)))
        cs = np.concatenate([[0], np.cumsum(sc)])
        for k in range(0, max(1, len(sc) - n + 1)):
            t_in = c['t'][k] - st / 2                                     # inicio útil de la toma
            t0 = t_in - PAD * speed
            if t0 < 0 or t0 + need > c['duration'] - 0.1: continue
            s_ = (cs[k + n] - cs[k]) / n
            for (uc, a, b) in used:
                if uc == ci:
                    s_ -= 0.6
                    if t_in < b and t_in + dur * speed > a: s_ -= 6        # nunca la misma parte de la toma
            if ci == prev_clip: s_ -= 1.5
            if best is None or s_ > best[0]: best = (s_, ci, t0, speed)
    if best is None:                                                       # tomas cortas: usa la más larga en bucle
        ci = max(range(len(clips)), key=lambda i: clips[i]['duration']); best = (0, ci, 0.0, 1.0)
    return best

def extract_shot(sh, fmt):
    c = clips[sh['clip']]; OW, OH = int(fmt['W'] * 1.08) // 2 * 2, int(fmt['H'] * 1.08) // 2 * 2
    sh['path'] = f"{WORK}/shots/{fmt['name']}_{sh['seg']:02d}_{sh['k']}.mp4"
    src_len = min(c['duration'] - sh['t0'], (sh['dur'] + 2 * PAD) * sh['speed'] + 0.3)
    vf = (tonemap(c) + f"setpts={1 / sh['speed']:.4f}*PTS,fps={FPS},"
          f"scale={OW}:{OH}:force_original_aspect_ratio=increase:flags=lanczos,crop={OW}:{OH},format=yuv420p")
    if not os.path.exists(sh['path']):
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-ss', f"{sh['t0']:.3f}", '-t', f"{src_len:.3f}", '-i', c['src'], '-vf', vf, '-an', *VENC, sh['path']])
        except RuntimeError:
            run(['ffmpeg', '-y', '-v', 'error', '-ss', f"{sh['t0']:.3f}", '-t', f"{src_len:.3f}", '-i', c['src'], '-vf', vf.replace(tonemap(c), ''),
                 '-an', *VENC, sh['path']])
    sh['amb'] = None
    if SONIDO_NATURAL and c['audio']:
        r = subprocess.run(['ffmpeg', '-v', 'error', '-ss', f"{sh['t0'] + PAD * sh['speed']:.3f}", '-t', f"{sh['dur'] + 1.0:.3f}", '-i', c['src'],
                            '-vn', '-ac', '2', '-ar', str(SR), '-f', 'f32le', '-'], capture_output=True)
        a = np.frombuffer(r.stdout, np.float32).reshape(-1, 2).copy()
        if len(a) and np.abs(a).max() > 1e-4: sh['amb'] = a

for fmt in FORMATOS:
    build_timeline(fmt)
    used, prev = [], -1
    q_es = clip_txt([s['visual'] for s in fmt['segs']]) if CLIP_M is not None else [None] * len(fmt['segs'])
    print(f"\n🎬 Tomas elegidas para {fmt['name']}:")
    for i, s in enumerate(fmt['segs']):
        shot_len = 5.5 if fmt['W'] > fmt['H'] else 3.6
        n = max(1, int(round(s['D'] / FPS / shot_len)))
        if s['tratamiento'] in ('periodico', 'titulo'): n = 1                 # una sola imagen de fondo
        elif s['tratamiento'] == 'pintura': n = min(n, 2)
        q_e = None
        if q_es[i] is not None:
            q_e = 0.8 * q_es[i] + 0.2 * GEN_E; q_e /= np.linalg.norm(q_e)
        s['shots'] = []; fb = s['D'] // n
        for k in range(n):
            a_ = k * fb; b_ = s['D'] if k == n - 1 else (k + 1) * fb
            dur = (b_ - a_) / FPS
            sc, ci, t0, speed = pick_shot(s['visual'], q_e, dur, used, prev)
            used.append((ci, t0 + PAD * speed, t0 + (PAD + dur) * speed)); prev = ci
            sh = {'seg': i, 'k': k, 'clip': ci, 't0': t0, 'speed': speed, 'dur': dur, 'a': a_, 'b': b_}
            extract_shot(sh, fmt); s['shots'].append(sh)
            print(f"  {i + 1:2d}.{k + 1} {clips[ci]['name'][:44]:44s} @{t0 + PAD * speed:6.1f}s  {dur:4.1f}s{' 🐢' if speed < 1 else ''}"
                  f"  ← \"{s['visual'][:40]}\"")

In [ ]:
#@title 7 · Animaciones: grafito, pintura animada, tipografía por época, calidad de imagen por época, periódico y transiciones del guion
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import urllib.request
FONT_DIR = f"{WORK}/fonts"
FAM = {'title': ('Cinzel', 700), 'serif': ('Playfair Display', 800), 'sans': ('Inter', 600), 'sub': ('Inter', 700),
       'hand': ('Caveat Brush', 400), 'mast': ('UnifrakturMaguntia', 400), 'news': ('Old Standard TT', 400), 'newsb': ('Old Standard TT', 700),
       'era80': ('Audiowide', 400), 'era90': ('VT323', 400), 'era00': ('Special Elite', 400), 'era10': ('Oswald', 600),
       'era17': ('Archivo Black', 400), 'era20': ('Space Grotesk', 600)}
FONT_FILE = {}
def fetch_font(role):
    fam, w = FAM[role]; dst = os.path.join(FONT_DIR, f"{fam.replace(' ', '')}_{w}.ttf")
    if os.path.exists(dst) and os.path.getsize(dst) > 1000: return dst
    for q in (f"{fam.replace(' ', '+')}:wght@{w}", fam.replace(' ', '+')):
        try:   # sin "User-Agent" de navegador, Google Fonts entrega .ttf
            css = urllib.request.urlopen(f"https://fonts.googleapis.com/css2?family={q}", timeout=15).read().decode()
            url = re.search(r'url\((https://[^)]+\.ttf)\)', css).group(1)
            urllib.request.urlretrieve(url, dst); return dst
        except Exception: pass
    return None
for r_ in FAM: FONT_FILE[r_] = fetch_font(r_)
print("Fuentes:", ', '.join(f"{r_}={'✓' if v else '—'}" for r_, v in FONT_FILE.items()))
# Tus fuentes: si el nombre lleva un año (ej. 1998_retro.ttf) se usan para esa década; 'titulo' en el nombre → títulos
fdir = os.path.join(PROYECTO, CARPETA_FUENTES)
USER_ERA_FONT = {}
if os.path.isdir(fdir):
    for f_ in os.listdir(fdir):
        if not f_.lower().endswith(('.ttf', '.otf')): continue
        p_ = os.path.join(fdir, f_); m_ = re.search(r'(19[5-9]\d|20[0-4]\d)', f_)
        if m_: USER_ERA_FONT[int(m_.group(1)) // 10 * 10] = p_
        if 'titulo' in f_.lower(): FONT_FILE['title'] = p_
    if USER_ERA_FONT: print("Tus fuentes por época:", {k: os.path.basename(v) for k, v in USER_ERA_FONT.items()})
FALLBACK = ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', '/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf']
_fc = {}
def font(role, size, path=None):
    key = (role, int(size), path)
    if key not in _fc:
        for cand in [path, FONT_FILE.get(role)] + FALLBACK:
            if not cand: continue
            try:
                f = ImageFont.truetype(cand, int(size))
                try: f.set_variation_by_axes([FAM.get(role, ('', 400))[1]])
                except Exception: pass
                _fc[key] = f; break
            except Exception: pass
        else: _fc[key] = ImageFont.load_default(size=int(size))
    return _fc[key]

def ERA(y):
    """Tipografía, calidad de imagen y animación de cada época."""
    if not y: return {'role': 'serif', 'look': None, 'anim': 'doc', 'color': (245, 245, 245), 'path': None}
    d = USER_ERA_FONT.get(y // 10 * 10)
    if y < 1990: e = {'role': 'era80', 'look': 'vhs', 'anim': 'vhs', 'color': (250, 140, 255)}
    elif y < 2000: e = {'role': 'era90', 'look': 'vhs', 'anim': 'vhs', 'color': (240, 240, 240)}
    elif y < 2010: e = {'role': 'era00', 'look': 'digicam', 'anim': 'typewriter', 'color': (225, 238, 245)}
    elif y < 2017: e = {'role': 'era10', 'look': 'hd', 'anim': 'news', 'color': (255, 255, 255)}
    elif y < 2021: e = {'role': 'era17', 'look': 'social', 'anim': 'glitch', 'color': (255, 255, 255)}
    else: e = {'role': 'era20', 'look': None, 'anim': 'clean', 'color': (255, 255, 255)}
    e['path'] = d; return e

def ease(x): x = min(1.0, max(0.0, x)); return x * x * (3 - 2 * x)
def ease_out(x): x = min(1.0, max(0.0, x)); return 1 - (1 - x) ** 3
_noise_cache = {}
def noise_field(W, H, scale, seed):
    key = (W, H, scale, seed)
    if key not in _noise_cache:
        rng = np.random.default_rng(seed); acc = np.zeros((H, W), np.float32); amp = 1.0; tot = 0
        for o in range(4):
            s = max(2, int(scale / 2 ** o)); n = rng.random((H // s + 2, W // s + 2)).astype(np.float32)
            acc += amp * cv2.resize(n, (W, H), interpolation=cv2.INTER_CUBIC); tot += amp; amp *= 0.5
        acc /= tot; _noise_cache[key] = (acc - acc.min()) / (np.ptp(acc) + 1e-6)
    return _noise_cache[key]

def paste_rgba(frame, rgba, x0, y0, alpha=1.0):
    h, w = rgba.shape[:2]; H, W = frame.shape[:2]
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + w), min(H, y0 + h)
    if xa >= xb or ya >= yb or alpha <= 0: return
    sub = rgba[ya - y0:yb - y0, xa - x0:xb - x0]; a = sub[..., 3:4].astype(np.float32) / 255 * alpha
    roi = frame[ya:yb, xa:xb]; roi[:] = (roi * (1 - a) + sub[..., :3] * a).astype(np.uint8)

# ---------- Texto animado por letra ----------
class TextBlock:
    """Texto renderizado una vez (con borde y sombra); guarda la caja de cada letra para animarlas una por una."""
    def __init__(self, text, role, size, color=(255, 255, 255), max_w=None, path=None, stroke=0, shadow=True, spacing=0, align='left'):
        f = font(role, size, path); self.size = size
        words = text.split(); lines, cur = [], ''
        for w in words:
            t_ = (cur + ' ' + w).strip()
            if max_w and cur and self._len(f, t_, spacing) > max_w: lines.append(cur); cur = w
            else: cur = t_
        if cur: lines.append(cur)
        asc, desc = f.getmetrics(); lh = int((asc + desc) * 1.08)
        widths = [self._len(f, l, spacing) for l in lines]; Wt = int(max(widths or [1])) + 2 * (stroke + 12); Ht = lh * len(lines) + 2 * (stroke + 12)
        img = Image.new('RGBA', (Wt, Ht), (0, 0, 0, 0)); d = ImageDraw.Draw(img); self.boxes = []
        for li, l in enumerate(lines):
            x = (Wt - widths[li]) / 2 if align == 'center' else stroke + 12; y = stroke + 12 + li * lh
            for ch in l:
                cw = f.getlength(ch)
                d.text((x, y), ch, font=f, fill=tuple(int(c) for c in color[::-1]) + (255,), stroke_width=stroke, stroke_fill=(0, 0, 0, 255))
                if ch.strip(): self.boxes.append((int(x) - stroke - 2, y - 4, int(x + cw) + stroke + 4, y + lh + 2))
                x += cw + spacing
        rgba = np.array(img)[..., [2, 1, 0, 3]]
        if shadow:
            sh = cv2.GaussianBlur(rgba[..., 3], (0, 0), max(2, size / 14)).astype(np.float32) * 0.75
            out = np.zeros_like(rgba); a_ = rgba[..., 3:4].astype(np.float32) / 255
            out[..., :3] = (rgba[..., :3] * a_).astype(np.uint8); out[..., 3] = np.maximum(rgba[..., 3], sh).astype(np.uint8)
            rgba = out
        self.rgba = rgba; self.w, self.h = rgba.shape[1], rgba.shape[0]; self.n = len(self.boxes)
    @staticmethod
    def _len(f, t, sp): return f.getlength(t) + sp * max(0, len(t) - 1)
    def frame(self, weights):
        """weights[i] = visibilidad de cada letra (0-1)."""
        m = np.zeros((self.h, self.w), np.float32)
        for (x0, y0, x1, y1), wv in zip(self.boxes, weights):
            if wv > 0: m[max(0, y0):y1, max(0, x0):x1] = np.maximum(m[max(0, y0):y1, max(0, x0):x1], wv)
        out = self.rgba.copy(); out[..., 3] = (out[..., 3] * m).astype(np.uint8); return out

_tb = {}; TXT_RECT = {}
def anim_text(fr, key, text, anim, t, dur, role, size, color, pos, path=None, max_w=None, align='left', stroke=0):
    """Dibuja texto animado según la época: doc (letra por letra), typewriter, vhs, news, glitch, clean."""
    if t < 0 or t > dur or not text: return 0
    k_ = (key, text, role, size, path, max_w, align, stroke)
    if k_ not in _tb: _tb[k_] = TextBlock(text, role, size, color, max_w, path, stroke=stroke, align=align, spacing=size * 0.02)
    tb = _tb[k_]; H, W = fr.shape[:2]
    out_a = 1 - ease((t - (dur - 0.45)) / 0.45)
    x = int(pos[0] - (tb.w / 2 if align == 'center' else 0)); y = int(pos[1] - tb.h / 2)
    TXT_RECT[key] = (x, y, tb.w, tb.h)
    if anim == 'typewriter':
        nvis = int(t * 16); wts = [1.0 if i < nvis else 0.0 for i in range(tb.n)]
        img = tb.frame(wts); paste_rgba(fr, img, x, y, out_a)
        if nvis < tb.n + 8 and int(t * 3) % 2 == 0 and tb.boxes:                    # cursor
            bx = tb.boxes[min(nvis, tb.n - 1)]; cx_ = x + (bx[2] if nvis >= tb.n else bx[0]); cv2.rectangle(fr, (cx_, y + bx[1] + 6), (cx_ + max(3, size // 10), y + bx[3] - 6), color, -1)
        return nvis
    if anim == 'vhs':
        on = 1.0 if t > 0.45 or random.random() > 0.35 else 0.0
        img = tb.frame([1.0] * tb.n); a = out_a * on
        red = img.copy(); red[..., :3] = (0, 0, 255); blue = img.copy(); blue[..., :3] = (255, 160, 0)
        paste_rgba(fr, red, x + 4, y, a * 0.55); paste_rgba(fr, blue, x - 4, y, a * 0.55); paste_rgba(fr, img, x, y, a)
        return 0
    if anim == 'news':
        e = ease_out(t / 0.4); bw = int((tb.w + 30) * e)
        if bw > 0:
            ov = fr.copy(); cv2.rectangle(ov, (x - 18, y + 6), (x - 18 + bw, y + tb.h - 6), (30, 30, 190), -1)
            cv2.addWeighted(ov, 0.85 * out_a, fr, 1 - 0.85 * out_a, 0, fr)
        img = tb.frame([1.0 if (b_[0] + 20) < bw else 0.0 for b_ in tb.boxes]); paste_rgba(fr, img, x, y, out_a); return 0
    if anim == 'glitch':
        img = tb.frame([1.0] * tb.n)
        if t < 0.4:
            for _ in range(6):
                y0 = random.randrange(0, max(1, tb.h - 8)); hh = random.randint(4, max(5, tb.h // 6))
                img[y0:y0 + hh] = np.roll(img[y0:y0 + hh], random.randint(-30, 30), axis=1)
            red = img.copy(); red[..., :3] = (60, 0, 255); paste_rgba(fr, red, x + 7, y, out_a * 0.6)
        paste_rgba(fr, img, x, y, out_a); return 0
    stagger = 0.035 if anim == 'doc' else 0.02
    wts = [ease((t - i * stagger) / 0.35) for i in range(tb.n)]
    paste_rgba(fr, tb.frame(wts), x, y + int(10 * (1 - ease(t / 0.5))), out_a); return 0

# ---------- Calidad de imagen de cada época ----------
_vhs = {}
def era_look(fr, look, t, year):
    H, W = fr.shape[:2]
    if look == 'vhs':
        s = cv2.resize(fr, (W // 5, H // 5), interpolation=cv2.INTER_AREA); f = cv2.resize(s, (W, H), interpolation=cv2.INTER_LINEAR)
        ycc = cv2.cvtColor(f, cv2.COLOR_BGR2YCrCb)
        for ch in (1, 2): ycc[..., ch] = np.roll(cv2.blur(ycc[..., ch], (15, 1)), 5, axis=1)
        f = cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR).astype(np.float32)
        f = f * 0.88 + 16 + np.random.normal(0, 7, (H // 2, W // 2, 1)).astype(np.float32).repeat(2, 0).repeat(2, 1)[:H, :W]
        if 'scan' not in _vhs or _vhs['scan'].shape[0] != H:
            _vhs['scan'] = np.where((np.arange(H) % 3) == 0, 0.86, 1.0).astype(np.float32)[:, None, None]
        f *= _vhs['scan']
        band_y = int((t * 0.23 % 1.3 - 0.15) * H); bh = int(H * 0.05)
        if 0 < band_y < H - bh:
            f[band_y:band_y + bh] = np.roll(f[band_y:band_y + bh], random.randint(8, 26), axis=1) + 25
        f = np.clip(f, 0, 255).astype(np.uint8)
        if W > H:                                                     # 4:3 de la época
            side = int((W - H * 4 / 3) / 2); f[:, :side] = 0; f[:, W - side:] = 0
        osd = 'era90'; sz = int(min(W, H) * 0.055)
        x0 = int((W - H * 4 / 3) / 2) + int(W * 0.03) if W > H else int(W * 0.06)
        anim_text(f, 'osd1', 'PLAY' if int(t * 2) % 2 == 0 or t > 2 else ' ', 'vhs', 1.0, 999, osd, sz, (240, 240, 240), (x0, int(H * 0.08)))
        anim_text(f, 'osd2', f"{year}", 'vhs', 1.0, 999, osd, sz, (240, 240, 240), (x0, int(H * 0.9)))
        return f
    if look == 'digicam':
        s = cv2.resize(fr, (W * 480 // min(W, H) // 2 * 2, H * 480 // min(W, H) // 2 * 2), interpolation=cv2.INTER_AREA)
        s = cv2.imdecode(cv2.imencode('.jpg', s, [cv2.IMWRITE_JPEG_QUALITY, 22])[1], 1)
        f = cv2.resize(s, (W, H), interpolation=cv2.INTER_LINEAR)
        hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV).astype(np.float32); hsv[..., 1] *= 1.2
        f = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)
        f = cv2.convertScaleAbs(f, alpha=1.08, beta=-6)
        sz = int(min(W, H) * 0.05)
        anim_text(f, 'dc', f"'{str(year)[2:]}", 'clean', 1.0, 999, 'era90', sz, (40, 140, 255), (int(W * 0.84), int(H * 0.9)))
        return f
    if look == 'hd':
        s = cv2.resize(fr, (W * 720 // min(W, H) // 2 * 2, H * 720 // min(W, H) // 2 * 2), interpolation=cv2.INTER_AREA)
        s = cv2.imdecode(cv2.imencode('.jpg', s, [cv2.IMWRITE_JPEG_QUALITY, 50])[1], 1)
        f = cv2.resize(s, (W, H), interpolation=cv2.INTER_LINEAR)
        return cv2.addWeighted(f, 1.35, cv2.GaussianBlur(f, (0, 0), 2), -0.35, 0)
    if look == 'social':
        return cv2.imdecode(cv2.imencode('.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 38])[1], 1)
    return fr

# ---------- Color documental ----------
def curve(points):
    xs, ys = zip(*points); return np.clip(np.interp(np.arange(256), xs, ys), 0, 255).astype(np.uint8)
LUT_B, LUT_G, LUT_R = curve([(0, 12), (64, 62), (128, 128), (200, 196), (255, 238)]), curve([(0, 6), (64, 60), (128, 128), (255, 246)]), curve([(0, 4), (64, 58), (128, 134), (200, 210), (255, 252)])
def grade_doc(fr, lv=None):
    if lv is not None:                                             # niveles automáticos (DJI plano / D-Log)
        lo, hi = lv; fr = cv2.convertScaleAbs(fr, alpha=225.0 / max(30, hi - lo), beta=-lo * 225.0 / max(30, hi - lo) + 8)
    b, g, r = cv2.split(fr); return cv2.merge([cv2.LUT(b, LUT_B), cv2.LUT(g, LUT_G), cv2.LUT(r, LUT_R)])

# ---------- Grafito (grabación real → dibujo a lápiz) ----------
_paper = {}
def paper(W, H, tint=(214, 226, 236)):
    key = (W, H, tint)
    if key not in _paper:
        n = noise_field(W, H, max(W, H) // 6, 3); fine = np.random.default_rng(4).normal(0, 4, (H, W)).astype(np.float32)
        p = np.array(tint, np.float32)[None, None] + ((n - 0.5) * 14 + fine)[..., None]
        _paper[key] = np.clip(p, 0, 255).astype(np.float32)
    return _paper[key]
_hatch = {}
def hatch(W, H):
    if (W, H) not in _hatch:
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        h1 = (np.sin((xx + yy) * 0.9) > 0.55).astype(np.float32); h2 = (np.sin((xx - yy) * 0.8) > 0.7).astype(np.float32)
        jit = noise_field(W, H, 9, 5); _hatch[(W, H)] = (h1 * (jit > 0.35), h2 * (jit > 0.5))
    return _hatch[(W, H)]
def graphite(fr):
    H, W = fr.shape[:2]; g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
    gf = g.astype(np.float32); blur = cv2.GaussianBlur(255 - gf, (0, 0), max(2.0, W / 260))
    dodge = np.clip(gf * 255 / (256 - blur), 0, 255) / 255
    e = cv2.GaussianBlur(cv2.Canny(cv2.GaussianBlur(g, (0, 0), 1.6), 35, 100).astype(np.float32) / 255, (0, 0), 0.8)
    shade = 1 - gf / 255; h1, h2 = hatch(W, H)
    tone = dodge * (1 - 0.45 * shade ** 1.3 * h1 - 0.35 * np.clip(shade - 0.45, 0, 1) * 2 * h2) - 0.5 * e
    tone = np.clip(tone, 0.08, 1)[..., None]
    out = paper(W, H) * tone; out[..., 0] *= 1.03                       # brillo plateado del grafito
    return np.clip(out, 0, 255).astype(np.uint8)
def graphite_reveal(color, drawing, p, seed=1):
    H, W = color.shape[:2]; n = noise_field(W, H, max(W, H) // 10, seed)
    key = ('ramp', W, H)
    if key not in _noise_cache:
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32); _noise_cache[key] = xx / W * 0.75 + yy / H * 0.25
    m = np.clip((p * 1.35 - _noise_cache[key] - 0.15 * n) / 0.07, 0, 1)[..., None]
    return (color * (1 - m) + drawing * m).astype(np.uint8)

# ---------- Pintura animada (estilo Veritasium: se pinta con pinceladas y se anota con pintura) ----------
def paint(fr):
    H, W = fr.shape[:2]; s = cv2.resize(fr, (640, int(640 * H / W) // 2 * 2), interpolation=cv2.INTER_AREA)
    st = cv2.stylization(s, sigma_s=45, sigma_r=0.4)
    st = ((st.astype(np.int16) // 20) * 20 + 10).clip(0, 255).astype(np.uint8)
    k = np.zeros((9, 9), np.float32); cv2.line(k, (0, 7), (8, 1), 1, 1); k /= k.sum()
    st = cv2.addWeighted(cv2.filter2D(st, -1, k), 0.6, st, 0.4, 0)
    up = cv2.resize(st, (W, H), interpolation=cv2.INTER_CUBIC).astype(np.float32)
    canvas = noise_field(W, H, 6, 9)[..., None]
    return np.clip(up * (0.9 + 0.16 * canvas), 0, 255).astype(np.uint8)
class BrushReveal:
    def __init__(self, W, H, seed):
        rng = random.Random(seed); self.W, self.H = W, H; self.strokes = []
        n = 22
        for i in range(n):
            y = (i + 0.5) / n * 1.1 - 0.05 + rng.uniform(-0.03, 0.03); ang = rng.uniform(-0.35, 0.35)
            x0, x1 = rng.uniform(-0.15, 0.1), rng.uniform(0.9, 1.15)
            pts = [(x0 + (x1 - x0) * u, y + math.tan(ang) * (u - 0.5) * 0.3 + 0.02 * math.sin(u * 6 + i)) for u in np.linspace(0, 1, 24)]
            self.strokes.append((pts, rng.uniform(0.07, 0.12), i / n * 0.8))
        self.bristle = noise_field(W, H, 3, seed + 11)
    def mask(self, p):
        w, h = self.W // 4, self.H // 4; m = np.zeros((h, w), np.float32)
        for pts, th, st in self.strokes:
            pr = min(1.0, max(0.0, (p - st) / 0.22))
            if pr <= 0: continue
            k = max(2, int(len(pts) * pr)); poly = np.array([(x * w, y * h) for x, y in pts[:k]], np.int32)
            cv2.polylines(m, [poly], False, 1.0, max(2, int(th * h)), cv2.LINE_AA)
        m = cv2.resize(cv2.GaussianBlur(m, (0, 0), 1.2), (self.W, self.H))
        return np.clip((m - 0.35 * self.bristle) * 2.2, 0, 1)
PAINT_COLOR = (40, 175, 250)                                            # ámbar
def paint_stroke_path(fr, pts, p, thick, color=PAINT_COLOR):
    if p <= 0 or len(pts) < 2: return
    k = max(2, int(len(pts) * min(1, p))); ov = fr.copy()
    for j in range(1, k):
        tj = thick * (0.6 + 0.4 * math.sin(math.pi * j / len(pts)))
        cv2.line(ov, tuple(map(int, pts[j - 1])), tuple(map(int, pts[j])), color, max(2, int(tj)), cv2.LINE_AA)
    cv2.addWeighted(ov, 0.92, fr, 0.08, 0, fr)
def annotation(fr, kind, p, anchor, seed=0):
    H, W = fr.shape[:2]; th = max(6, int(min(W, H) * 0.014)); rng = random.Random(seed)
    if kind == 'circulo':
        cx, cy = W * 0.5, H * 0.42; rx, ry = W * (0.2 if W > H else 0.32), H * (0.2 if W > H else 0.12)
        pts = [(cx + rx * (1 + 0.05 * math.sin(a * 3)) * math.cos(a), cy + ry * math.sin(a)) for a in np.linspace(-1.9, -1.9 + 2 * math.pi * 1.06, 70)]
        paint_stroke_path(fr, pts, p, th)
    elif kind == 'subrayado' and anchor:
        x0, y0, w0 = anchor; pts = [(x0 + w0 * u, y0 + 6 * math.sin(u * 9 + seed)) for u in np.linspace(0, 1, 40)]
        paint_stroke_path(fr, pts, p, th)
    elif kind == 'flecha' and anchor:
        x0, y0, w0 = anchor; sx, sy = x0 + w0 * 0.5, y0 + 10; ex, ey = W * 0.5, H * 0.38
        pts = [(sx + (ex - sx) * u + math.sin(u * math.pi) * W * 0.06, sy + (ey - sy) * u) for u in np.linspace(0, 1, 40)]
        paint_stroke_path(fr, pts, p, th)
        if p > 0.95:
            a = math.atan2(ey - pts[-3][1], ex - pts[-3][0])
            for da in (2.5, -2.5):
                cv2.line(fr, (int(ex), int(ey)), (int(ex - 50 * math.cos(a + da * 0.2)), int(ey - 50 * math.sin(a + da * 0.2))), PAINT_COLOR, th, cv2.LINE_AA)

# ---------- Escena de periódico ----------
def halftone(gray, cell=7):
    H, W = gray.shape; yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    a = math.radians(45); u = (xx * math.cos(a) + yy * math.sin(a)) * (2 * math.pi / cell); v = (-xx * math.sin(a) + yy * math.cos(a)) * (2 * math.pi / cell)
    screen = (np.cos(u) * np.cos(v) + 1) / 2
    return np.where(gray.astype(np.float32) / 255 > screen * 0.9 + 0.05, 255, 30).astype(np.uint8)
def build_page(s, photo, body_text):
    PW, PH = 1500, 2000; img = Image.new('RGB', (PW, PH), (236, 226, 205)); d = ImageDraw.Draw(img); ink = (28, 26, 24)
    mast = "La Gaceta del Fandom"; fm = font('mast', 128)
    d.text(((PW - fm.getlength(mast)) / 2, 50), mast, font=fm, fill=ink)
    d.line((60, 215, PW - 60, 215), fill=ink, width=4); d.line((60, 224, PW - 60, 224), fill=ink, width=1)
    dl = f"EDICIÓN ESPECIAL  ·  {s.get('lugar') or 'ESTADOS UNIDOS'}  ·  {s['anio']}  ·  RECREACIÓN".upper(); fd = font('news', 30)
    d.text(((PW - fd.getlength(dl)) / 2, 234), dl, font=fd, fill=ink); d.line((60, 280, PW - 60, 280), fill=ink, width=2)
    head = (s.get('titular') or s.get('texto_pantalla') or '').upper(); size = 120
    while size > 50:
        fh = font('serif', size); words = head.split(); lines, cur = [], ''
        for w in words:
            t_ = (cur + ' ' + w).strip()
            if cur and fh.getlength(t_) > PW - 140: lines.append(cur); cur = w
            else: cur = t_
        if cur: lines.append(cur)
        if len(lines) <= 3: break
        size -= 8
    y = 305; asc, desc = fh.getmetrics()
    for l in lines: d.text(((PW - fh.getlength(l)) / 2, y), l, font=fh, fill=ink); y += int((asc + desc) * 1.0)
    head_box = (305, y); y += 25
    ph_h = 620; g = cv2.cvtColor(cv2.resize(photo, (PW - 140, ph_h)), cv2.COLOR_BGR2GRAY)
    g = cv2.convertScaleAbs(g, alpha=1.25, beta=-20)
    img.paste(Image.fromarray(halftone(g)).convert('RGB'), (70, y)); d.rectangle((70, y, PW - 70, y + ph_h), outline=ink, width=3)
    y += ph_h + 10; fc = font('news', 26); d.text((70, y), "Imagen de archivo. Recreación para el documental.", font=fc, fill=ink); y += 50
    fb = font('news', 29); colw = (PW - 140 - 2 * 40) // 3; words = (body_text + ' ') * 6; words = words.split()
    for col in range(3):
        x = 70 + col * (colw + 40); yy_ = y; line = ''
        while words and yy_ < PH - 80:
            t_ = (line + ' ' + words[0]).strip()
            if fb.getlength(t_) > colw: d.text((x, yy_), line, font=fb, fill=ink); yy_ += 37; line = ''
            else: line = t_; words.pop(0)
        if col < 2: d.line((x + colw + 20, y, x + colw + 20, PH - 80), fill=ink, width=1)
    page = np.array(img)[..., ::-1].astype(np.float32)
    page *= (0.93 + 0.1 * noise_field(PW, PH, 300, 21))[..., None]; page += np.random.default_rng(2).normal(0, 5, page.shape[:2])[..., None]
    page[PH // 2 - 3:PH // 2 + 1] *= 0.82; page[PH // 2 + 1:PH // 2 + 5] *= 1.05            # doblez
    page = cv2.GaussianBlur(np.clip(page, 0, 255).astype(np.uint8), (0, 0), 0.7)
    return page, head_box
class Newspaper:
    def __init__(self, s, photo, body, W, H):
        self.page, (hy0, hy1) = build_page(s, photo, body); self.W, self.H = W, H
        PH, PW = self.page.shape[:2]; self.fit = (0.9 * H / PH) if W > H else (0.95 * W / PW)
        self.head_y = (hy0 + hy1) / 2; self.ones = np.full((PH, PW), 255, np.uint8)
    def draw(self, bg, t, D):
        W, H = self.W, self.H; PH, PW = self.page.shape[:2]
        out = (cv2.GaussianBlur(bg, (0, 0), 9) * 0.32).astype(np.uint8)
        e = ease_out(t / 1.25); ang = (1 - e) * 540 - 2.5 * e; sc = self.fit * (0.06 + 0.94 * e)
        u = ease((t - 1.25) / max(0.1, D - 1.25)); sc *= 1 + (0.35 if W > H else 0.1) * u                       # luego se acerca al titular
        cy_page = PH / 2 + (self.head_y - PH / 2) * u * 0.85
        M = cv2.getRotationMatrix2D((PW / 2, cy_page), ang, sc); M[0, 2] += W / 2 - PW / 2; M[1, 2] += H / 2 - cy_page
        warped = cv2.warpAffine(self.page, M, (W, H), flags=cv2.INTER_LINEAR); m = cv2.warpAffine(self.ones, M, (W, H))
        shadow = cv2.GaussianBlur(np.roll(np.roll(m, 22, 0), 16, 1), (0, 0), 18).astype(np.float32)[..., None] / 255
        out = (out * (1 - 0.6 * shadow)).astype(np.uint8); a = m.astype(np.float32)[..., None] / 255
        return (out * (1 - a) + warped * a).astype(np.uint8)

# ---------- Transiciones del guion ----------
_dots = {}
def dot_order(W, H):
    if (W, H) not in _dots:
        rng = np.random.default_rng(8); O = np.full((H, W), 2.0, np.float32); n = int(W * H / 140)
        xs, ys = rng.integers(0, W, n), rng.integers(0, H, n); rs = rng.integers(2, 6, n); vs = rng.random(n).astype(np.float32)
        for x, y, r, v in zip(xs, ys, rs, vs): cv2.circle(O, (int(x), int(y)), int(r), float(v), -1)
        _dots[(W, H)] = O
    return _dots[(W, H)]
def edge_field(W, H):
    key = ('edge', W, H)
    if key not in _noise_cache:
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        e = np.minimum(np.minimum(xx, W - 1 - xx) / (W / 2), np.minimum(yy, H - 1 - yy) / (H / 2))
        _noise_cache[key] = 0.72 * e + 0.28 * noise_field(W, H, max(W, H) // 14, 31) + 0.04 * noise_field(W, H, 4, 32)
    return _noise_cache[key]
def transition(kind, A, B, p, g=0):
    H, W = A.shape[:2]
    if kind == 'desgarre':                          # los bordes se desgastan como papel lijado
        f = edge_field(W, H); th = p * 1.08
        m = (f < th).astype(np.float32); rim = ((f >= th) & (f < th + 0.025)).astype(np.float32)
        m = cv2.GaussianBlur(m, (0, 0), 0.8)[..., None]; rim = cv2.GaussianBlur(rim, (0, 0), 1.0)[..., None]
        Af = A.astype(np.float32); Af = Af * (1 - rim) + (paper(W, H) * 0.95) * rim
        return (Af * (1 - m) + B * m).astype(np.uint8)
    if kind == 'enfoque_grafito':                   # rack focus: el desenfoque se vuelve puntos de grafito
        Bg = graphite(B); O = dot_order(W, H)
        if p < 0.5:
            base = cv2.GaussianBlur(A, (0, 0), 1 + 22 * p); dm = (O < p * 1.3).astype(np.float32)[..., None]
            return (base * (1 - dm) + Bg * dm).astype(np.uint8)
        q = (p - 0.5) / 0.5
        base = cv2.GaussianBlur(A, (0, 0), 12); dm = np.clip((O < 0.65 + q * 1.4).astype(np.float32), 0, 1)[..., None]
        mid = (base * (1 - dm) + Bg * dm)
        c = ease((q - 0.45) / 0.55); Bs = cv2.GaussianBlur(B, (0, 0), 0.1 + 6 * (1 - c)) if c < 1 else B
        return (mid * (1 - c) + Bs * c).astype(np.uint8)
    if kind == 'tunel':                             # obsesión: entramos por la mirada hacia la siguiente escena
        e = ease(p); z = 1 + 3.2 * e ** 1.6
        def zoom(img, s):
            M = cv2.getRotationMatrix2D((W / 2, H / 2), 0, s); return cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT)
        Az = cv2.addWeighted(zoom(A, z), 0.5, zoom(A, z * 0.94), 0.5, 0)
        r = ease((p - 0.3) / 0.7) * math.hypot(W, H) * 0.55
        key = ('rad', W, H)
        if key not in _noise_cache:
            yy, xx = np.mgrid[0:H, 0:W].astype(np.float32); _noise_cache[key] = np.hypot(xx - W / 2, yy - H / 2)
        d = _noise_cache[key]; m = np.clip((r - d) / (40 + 0.15 * r), 0, 1)[..., None]
        ring = np.clip(1 - np.abs(d - r) / 60, 0, 1)[..., None] * (1 if r > 1 else 0)
        Bz = zoom(B, 0.75 + 0.25 * e) if e < 1 else B
        out = Az * (1 - m) + Bz * m; out *= (1 - 0.7 * ring)
        return np.clip(out, 0, 255).astype(np.uint8)
    if kind == 'luz_alta':                          # quemado a blanco
        if p < 0.5:
            q = ease(p / 0.5); Af = A.astype(np.float32) * (1 + 3 * q) + 255 * q * 0.6
            bloom = cv2.GaussianBlur(np.clip(Af, 0, 255).astype(np.uint8), (0, 0), 25).astype(np.float32)
            return np.clip(Af * 0.6 + bloom * 0.6, 0, 255).astype(np.uint8)
        q = ease((p - 0.5) / 0.5); return np.clip(255 * (1 - q) + B.astype(np.float32) * q, 0, 255).astype(np.uint8)
    if kind == 'corte_accion':                      # sacudida tras el portazo
        q = 1 - p; dx, dy = int(random.uniform(-1, 1) * 30 * q), int(random.uniform(-1, 1) * 22 * q)
        M = np.float32([[1, 0, dx], [0, 1, dy]]); out = cv2.warpAffine(B, M, (W, H), borderMode=cv2.BORDER_REFLECT)
        return out if p > 0.15 else cv2.convertScaleAbs(out, alpha=0.6)
    return cv2.addWeighted(A, 1 - ease(p), B, ease(p), 0)       # disolver
def memoria_corrupta(frozen, q, g):
    """La imagen se congela y se corrompe (bloques, colores, desgarros) mientras el audio sigue distorsionado."""
    H, W = frozen.shape[:2]; f = frozen.copy(); rng = random.Random(g); n = int(4 + 60 * q ** 1.5); bs = 32
    for _ in range(n):
        x, y = rng.randrange(0, W - bs * 3), rng.randrange(0, H - bs)
        w = min(bs * rng.randint(1, 6), W - x); sx = min(W - w, max(0, x + rng.randint(-120, 120))); f[y:y + bs, x:x + w] = frozen[y:y + bs, sx:sx + w]
    if q > 0.2:
        b, g_, r = cv2.split(f); s_ = int(4 + 20 * q); f = cv2.merge([np.roll(b, -s_, 1), g_, np.roll(r, s_, 1)])
    if q > 0.35:
        lv = max(2, int(10 - 8 * q)); f = ((f // (256 // lv)) * (256 // lv)).astype(np.uint8)
    for _ in range(int(3 + 10 * q)):
        y = rng.randrange(0, H - 14); hh = rng.randint(2, 12); f[y:y + hh] = np.roll(f[y:y + hh], rng.randint(-200, 200), axis=1)
    if q > 0.88: f = (f * (1 - (q - 0.88) / 0.12)).astype(np.uint8)
    return f

# ---------- Acabado: grano de película, viñeta, barras ----------
_fin = {}
def finish(fr, bars=False):
    H, W = fr.shape[:2]
    if (W, H) not in _fin:
        yy, xx = np.mgrid[0:H, 0:W]; d = np.sqrt(((xx - W / 2) / (W / 2)) ** 2 + ((yy - H / 2) / (H / 2)) ** 2)
        _fin[(W, H)] = ((1 - 0.32 * np.clip(d - 0.45, 0, 1) ** 1.5)[..., None].astype(np.float32),
                        [np.random.default_rng(k).normal(0, 4.5, (H // 2, W // 2)).astype(np.float32) for k in range(6)])
    vig, grains = _fin[(W, H)]; gr = cv2.resize(grains[random.randrange(6)], (W, H))[..., None]
    out = np.clip(fr.astype(np.float32) * vig + gr, 0, 255).astype(np.uint8)
    if bars:
        bh = int((H - W / 2.39) / 2); out[:bh] = 0; out[H - bh:] = 0
    return out
print("Librería de animación lista ✓")

In [ ]:
#@title 8 · Composición: cada segmento con su tratamiento, textos por época, transiciones y subtítulos
class ShotReader:
    def __init__(self, path): self.path, self.cap, self.idx, self.last = path, None, -1, None
    def get(self, j):
        if self.cap is None or j < self.idx:
            if self.cap is not None: self.cap.release()
            self.cap = cv2.VideoCapture(self.path); self.idx = -1
        while self.idx < j:
            ok, fr = self.cap.read()
            if not ok: self.idx = j; break
            self.last = fr; self.idx += 1
        return self.last
    def close(self):
        if self.cap is not None: self.cap.release(); self.cap = None

class SegRender:
    def __init__(self, fmt, s, k):
        self.fmt, self.s, self.k = fmt, s, k; self.W, self.H = fmt['W'], fmt['H']
        self.readers = {}; self.cache = {}; self.news = None; self.brush = None; self.era = ERA(s['era'])
        self.padf = int(PAD * FPS)
    def close(self):
        for r in self.readers.values(): r.close()
        self.readers = {}
    def _raw(self, sh, local):
        r = self.readers.setdefault(id(sh), ShotReader(sh['path']))
        fr = r.get(max(0, local + self.padf))
        if fr is None: fr = np.zeros((int(self.H * 1.08) // 2 * 2, int(self.W * 1.08) // 2 * 2, 3), np.uint8)
        if 'lv' not in sh:
            g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY); lo, hi = np.percentile(g, 1), np.percentile(g, 99.5)
            sh['lv'] = (float(lo), float(hi)) if COLOR_AUTOMATICO and (hi - lo) < 200 else None    # sólo si la toma está "plana"
        BH, BW = fr.shape[:2]; n = max(1, sh['b'] - sh['a']); u = min(1.0, max(0.0, local / n))
        zz = u if sh['k'] % 2 == 0 else 1 - u
        cw = BW - (BW - self.W) * zz; ch = cw * self.H / self.W
        cx = BW / 2 + (u - 0.5) * (BW - cw) * (0.6 if sh['seg'] % 2 else -0.6); cy = BH / 2
        x0 = int(np.clip(cx - cw / 2, 0, BW - cw)); y0 = int(np.clip(cy - ch / 2, 0, BH - ch))
        crop = fr[y0:y0 + int(ch), x0:x0 + int(cw)]
        return grade_doc(cv2.resize(crop, (self.W, self.H), interpolation=cv2.INTER_AREA), sh['lv'])
    def base(self, j):
        shots = self.s['shots']; k = 0
        while k < len(shots) - 1 and j >= shots[k]['b']: k += 1
        sh = shots[k]; f = self._raw(sh, j - sh['a'])
        if k + 1 < len(shots) and j >= shots[k + 1]['a'] - 6:                 # mini disolvencia entre tomas
            nx = shots[k + 1]; a = (j - (nx['a'] - 6)) / 6
            f = cv2.addWeighted(f, 1 - a, self._raw(nx, j - nx['a']), a, 0)
        return f
    def frame(self, j):
        s, W, H = self.s, self.W, self.H; t = j / FPS; D = s['D'] / FPS; tr = s['tratamiento']; era = self.era; vert = H > W
        f = self.base(j)
        cr = next((c_ for c_ in s.get('creditos', []) if c_['a'] - 2 <= j < c_['b']), None)
        if cr is not None:                                                   # crédito impreso en el piso (antes del grafito/pintura: se vuelve parte del dibujo)
            if cr.get('obj') is None: cr['obj'] = FloorText(cr['text'], W, H, self.k, small=cr.get('final', False))
            cr['obj'].update(f, j)
            if j >= cr['a'] and tr != 'titulo': cr['obj'].draw(f, (j - cr['a']) / FPS, (cr['b'] - cr['a']) / FPS)
        if era['look'] and tr in ('normal', 'anio', 'grafito'): f = era_look(f, era['look'], t, s['era'])
        if tr == 'grafito':
            p = (t - min(0.25 * D, 2.0)) / 1.6
            if p > 0: f = graphite(f) if p > 1.4 else graphite_reveal(f, graphite(f), p, seed=self.k)
        elif tr == 'pintura':
            key = j // 3                                                       # pintura "a tres": se siente hecha a mano
            if self.cache.get('pk') != key: self.cache['pk'] = key; self.cache['paint'] = paint(f)
            painted = self.cache['paint']; p = t / 1.3
            if p < 1.08:
                if self.brush is None: self.brush = BrushReveal(W, H, self.k)
                m = self.brush.mask(p)[..., None]; f = (paper(W, H) * (1 - m) + painted * m).astype(np.uint8)
            else: f = painted
        elif tr == 'periodico':
            if self.news is None:
                body = ' '.join(x['texto'] for x in self.fmt['segs'] if x['anio'] == s['anio']) or s['texto']
                self.news = Newspaper(s, f, body, W, H)
            f = self.news.draw(f, t, D)
        elif tr == 'titulo':
            f = (f.astype(np.float32) * 0.45).astype(np.uint8)
            if cr is not None and j >= cr['a']: cr['obj'].draw(f, (j - cr['a']) / FPS, (cr['b'] - cr['a']) / FPS)   # crédito final, brillante
        # --- textos ---
        lead = s['lead']; txt = (s.get('texto_pantalla') or '').strip()
        if tr == 'anio' and s['anio']:
            big = int((H * 0.26) if not vert else W * 0.3)
            anim_text(f, ('y', self.k), str(s['anio']), era['anim'], t - 0.15, lead + 1.1, era['role'], big, era['color'], (W / 2, H * 0.45),
                      path=era['path'], align='center')
            if s.get('lugar'):
                anim_text(f, ('l', self.k), s['lugar'].upper(), 'doc' if era['anim'] in ('vhs', 'glitch') else era['anim'], t - 0.6, lead + 0.7,
                          era['role'], int(big * 0.18), era['color'], (W / 2, H * 0.45 + big * 0.62), path=era['path'], align='center')
        if tr == 'titulo' and txt:
            size = int(H * 0.1) if not vert else int(W * 0.1)
            anim_text(f, ('T', self.k), txt, 'doc', t - lead * 0.4, D + 5, 'title', size, (225, 235, 245), (W / 2, H * 0.47),
                      max_w=W * 0.86, align='center')
            r_ = TXT_RECT.get(('T', self.k))
            if r_ and t > lead * 0.4 + 0.6:
                e = ease_out((t - lead * 0.4 - 0.6) / 1.2); half = int(r_[2] * 0.4 * e)
                cv2.line(f, (W // 2 - half, r_[1] + r_[3] + 6), (W // 2 + half, r_[1] + r_[3] + 6), (90, 185, 230), max(2, H // 400), cv2.LINE_AA)
        elif txt and tr != 'periodico':
            t0 = (lead + 1.1) if tr == 'anio' else (1.4 if tr == 'pintura' else (min(0.25 * D, 2.0) + 0.9 if tr == 'grafito' else max(0.5, lead * 0.6)))
            if tr == 'pintura': role, anim, path, col = 'hand', 'doc', None, (255, 255, 255)
            elif s['era']: role, anim, path, col = era['role'], era['anim'], era['path'], era['color']
            else: role, anim, path, col = 'serif', 'doc', None, (245, 245, 245)
            if tr == 'grafito': col = (48, 46, 44)                  # tinta de grafito sobre el papel
            size = int(H * 0.075) if not vert else int(W * 0.085)
            pos, align = ((W * 0.07, H * 0.8), 'left') if not vert else ((W / 2, H * 0.22), 'center')
            if tr == 'pintura' and not vert: pos = (W * 0.07, H * 0.78)
            if s.get('creditos') and not vert: pos = (W * 0.07, H * 0.14)          # el piso es de los créditos
            anim_text(f, ('x', self.k), txt, anim, t - t0, D - t0 - 0.2, role, size, col, pos, path=path, max_w=W * 0.8, align=align, stroke=0)
            r_ = TXT_RECT.get(('x', self.k)); kind = s.get('anotacion', 'ninguna')
            if kind != 'ninguna' and tr in ('pintura', 'grafito') and r_ and t > t0 + 0.6 and t < D - 0.3:
                annotation(f, kind, (t - t0 - 0.6) / 0.9, (r_[0] + 14, r_[1] + r_[3] - 8, r_[2] - 28), seed=self.k)
        elif tr == 'pintura' and s.get('anotacion') == 'circulo' and t > 1.6:
            annotation(f, 'circulo', (t - 1.6) / 0.9, None, seed=self.k)
        return f

# ---------- Créditos impresos en el piso ----------
def credit_font_path():
    if FUENTE_CREDITOS.strip():
        for d_ in (os.path.join(PROYECTO, CARPETA_FUENTES), PROYECTO):
            p_ = os.path.join(d_, FUENTE_CREDITOS.strip())
            if os.path.exists(p_): return p_
        print(f"⚠ No encuentro la fuente {FUENTE_CREDITOS}; uso Cinzel.")
    return None
CRED_FONT = credit_font_path()
class FloorText:
    """Texto 'impreso' en el suelo: se coloca en perspectiva en la parte baja de la toma y se pega al piso con
    seguimiento de puntos (flujo óptico + homografía), así se mueve con la cámara/dron como si estuviera pintado ahí."""
    def __init__(self, text, W, H, seed, small=False):
        self.W, self.H = W, H; vert = H > W; self.bright = small
        size = int((W * 0.085) if vert else (H * 0.11)) * (0.8 if small else 1)
        tb = TextBlock(text, 'title', int(size), (240, 244, 246), max_w=W * (0.92 if vert else (0.8 if small else 0.7)), path=CRED_FONT, shadow=False, align='center', spacing=size * 0.06)
        self.rgba = tb.rgba; th, tw = self.rgba.shape[:2]
        grit = noise_field(tw, th, 3, seed + 40) * 0.6 + noise_field(tw, th, 14, seed + 41) * 0.4      # pintura gastada sobre la tierra
        self.rgba[..., 3] = (self.rgba[..., 3] * np.clip((grit - 0.08) * 2.6, 0, 1)).astype(np.uint8)
        near_y = H * (0.95 if vert else 0.93); near_w = W * (0.9 if vert else 0.78); far_w = near_w * 0.6
        depth = near_w * th / tw * 0.6
        src = np.float32([[0, 0], [tw, 0], [tw, th], [0, th]])
        dst = np.float32([[W / 2 - far_w / 2, near_y - depth], [W / 2 + far_w / 2, near_y - depth], [W / 2 + near_w / 2, near_y], [W / 2 - near_w / 2, near_y]])
        self.Hb = cv2.getPerspectiveTransform(src, dst); self.Hacc = np.eye(3); self.prev = None; self.pts = None; self.last_j = None
        self.mask = np.zeros((H // 2, W // 2), np.uint8); self.mask[int(H * 0.25):, :] = 255             # buscar puntos en el suelo (mitad de abajo)
    def update(self, fr, j):
        g = cv2.cvtColor(cv2.resize(fr, (self.W // 2, self.H // 2)), cv2.COLOR_BGR2GRAY)
        if self.prev is not None and self.last_j == j - 1 and self.pts is not None and len(self.pts) >= 8:
            nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, g, self.pts, None, winSize=(21, 21), maxLevel=3)
            ok = st.ravel() == 1
            if ok.sum() >= 8:
                Hm, inl = cv2.findHomography(self.pts[ok], nxt[ok], cv2.RANSAC, 2.0)
                if Hm is not None and int(inl.sum()) >= 8:
                    c4 = np.float32([[0, 0], [self.W / 2, 0], [self.W / 2, self.H / 2], [0, self.H / 2]]).reshape(-1, 1, 2)
                    if np.abs(cv2.perspectiveTransform(c4, Hm) - c4).max() > 40: Hm = None     # movimiento imposible: lo ignoro (agua, cielo, ruido)
                if Hm is not None and int(inl.sum()) >= 8:
                    S = np.diag([2.0, 2.0, 1.0]); self.Hacc = S @ Hm @ np.linalg.inv(S) @ self.Hacc
                self.pts = nxt[ok].reshape(-1, 1, 2)
        if self.pts is None or len(self.pts) < 40 or (j % 12 == 0):
            self.pts = cv2.goodFeaturesToTrack(g, 300, 0.01, 8, mask=self.mask)
        self.prev = g; self.last_j = j
    def draw(self, fr, t, dur):
        W, H = self.W, self.H; M = self.Hacc @ self.Hb
        warped = cv2.warpPerspective(self.rgba, M, (W, H), flags=cv2.INTER_LINEAR)
        a = warped[..., 3].astype(np.float32) / 255
        if a.max() <= 0: return
        rev = ease(t / 1.0)                                                       # se "imprime" de izquierda a derecha
        cols = np.where(a.max(0) > 0)[0]
        if rev < 1 and len(cols):
            x_rev = cols[0] + (cols[-1] - cols[0]) * rev
            a *= np.clip((x_rev - np.arange(W)) / 30, 0, 1)[None, :]
        a *= 1 - ease((t - (dur - 0.6)) / 0.6)
        a = cv2.GaussianBlur(a, (0, 0), 0.7)[..., None]
        lum = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY).astype(np.float32)[..., None] / 255
        ink = warped[..., :3].astype(np.float32) * ((1.0 + 0.15 * lum) if self.bright else (0.7 + 0.45 * lum))            # la pintura toma la luz y la textura del suelo
        halo = cv2.GaussianBlur(a[..., 0], (0, 0), 6)[..., None]                 # mancha oscura alrededor: se lee sobre cualquier suelo
        base = fr.astype(np.float32) * (1 - 0.45 * np.clip(halo * 1.5, 0, 1))
        fr[:] = np.clip(base * (1 - a * 0.95) + ink * a * 0.95, 0, 255).astype(np.uint8)

def load_credits():
    p_ = os.path.join(PROYECTO, 'creditos.txt')
    if os.path.exists(p_):
        return [l.strip() for l in open(p_, encoding='utf-8', errors='ignore') if l.strip()]
    return [c.strip() for c in CREDITOS.split('|') if c.strip()]
CRED_LIST = load_credits()
def assign_credits(fmt):
    for s in fmt['segs']: s['creditos'] = []
    if not (CRED_LIST or CREDITO_FINAL.strip()) or (fmt['H'] > fmt['W'] and not CREDITOS_EN_TIKTOK): return
    ok = [s for s in fmt['segs'] if s['tratamiento'] in ('normal', 'anio', 'grafito', 'pintura')]
    if not ok: return
    if CREDITOS_DONDE == 'repartidos':
        step = max(1, len(ok) // max(1, len(CRED_LIST))); ok = ok[::step]
    k = 0
    for s in ok:
        if k >= len(CRED_LIST): break
        per = 2 if len(CRED_LIST) - k > len(ok) - ok.index(s) else 1             # si no alcanzan los segmentos, 2 por segmento
        start = int((s['lead'] + 1.2 + (2.2 if s['tratamiento'] == 'anio' else 0)) * FPS)
        span = min(s['D'] - start - int(0.4 * FPS), int(per * 4.5 * FPS))
        if span < 2 * FPS: continue
        for q in range(per):
            if k >= len(CRED_LIST): break
            a = start + q * span // per; b = start + (q + 1) * span // per
            s['creditos'].append({'text': CRED_LIST[k].upper(), 'a': a, 'b': b}); k += 1
    last = fmt['segs'][-1]
    if CREDITO_FINAL.strip():                                                   # "una producción de…" al cierre, bajo el título
        a = int((last['lead'] + last['vd'] + 0.2) * FPS); b = last['D'] - int(0.2 * FPS)       # cuando ya terminó la voz
        if b - a < int(2.5 * FPS): a = max(0, b - int(2.5 * FPS))
        last['creditos'].append({'text': CREDITO_FINAL.strip().upper(), 'a': a, 'b': b, 'final': True})
    if k < len(CRED_LIST): print(f"⚠ {fmt['name']}: sólo cupieron {k} de {len(CRED_LIST)} créditos")
    print(f"🎬 Créditos en el piso ({fmt['name']}):", ', '.join(c_['text'] for s in fmt['segs'] for c_ in s['creditos']))
if CRED_LIST or CREDITO_FINAL.strip():
    for fmt in FORMATOS: assign_credits(fmt)
else:
    print("(Sin créditos en el piso: escribe CREDITOS en la celda 2 o sube creditos.txt)")

def make_subs(fmt):
    subs = []
    if not fmt['subs']: return subs
    for s in fmt['segs']:
        words = s['texto'].replace('«', '').replace('»', '').split(); chunks, cur = [], []
        for w in words:
            cur.append(w)
            if len(cur) >= 5 or re.search(r'[.,;:!?…]$', w): chunks.append(' '.join(cur)); cur = []
        if cur: chunks.append(' '.join(cur))
        tot = sum(len(c) for c in chunks) or 1; f = s['voz_f']; span = s['vd'] * FPS
        for c in chunks:
            n = span * len(c) / tot; subs.append((int(f), int(f + n), c)); f += n
    return subs
_subs_tb = {}
def draw_sub(fr, txt):
    H, W = fr.shape[:2]; vert = H > W
    if txt not in _subs_tb:
        _subs_tb[txt] = TextBlock(txt, 'sub', int(W * 0.058) if vert else int(H * 0.042), (255, 255, 255), max_w=W * 0.84, stroke=3, align='center')
    tb = _subs_tb[txt]; paste_rgba(fr, tb.rgba, int(W / 2 - tb.w / 2), int((H * 0.7 if vert else H * 0.88) - tb.h / 2))

def render_frames(fmt, emit, only=None):
    """Recorre toda la línea de tiempo y llama emit(g, frame). only = conjunto de frames (para la vista previa)."""
    segs = fmt['segs']; N = len(segs); g = 0; R = [SegRender(fmt, s, k) for k, s in enumerate(segs)]
    def want(n): return only is None or any(g <= x < g + n for x in only)
    for k, s in enumerate(segs):
        a0 = segs[k - 1]['post'] if k > 0 else 0; a1 = s['D'] - (s['pre'] if k < N - 1 else 0)
        for j in range(a0, a1):
            if only is None or g in only: emit(g, R[k].frame(j))
            g += 1
        if k < N - 1:
            nx = segs[k + 1]
            if s['transicion'] == 'memoria_corrupta':
                if want(s['gap']): frozen = R[k].frame(s['D'] - 1)
                for q in range(s['gap']):
                    if only is None or g in only: emit(g, memoria_corrupta(frozen, q / max(1, s['gap']), g))
                    g += 1
            else:
                n = s['pre'] + s['post']
                for i in range(n):
                    if only is None or g in only:
                        B = R[k + 1].frame(-s['pre'] + i)
                        A = B if s['transicion'] == 'corte_accion' else R[k].frame(s['D'] - s['pre'] + i)
                        emit(g, transition(s['transicion'], A, B, (i + 0.5) / n, g))
                    g += 1
        R[k].close()
    return g

def post(fmt, g, fr, subs):
    for a, b, txt in subs:
        if a <= g < b: draw_sub(fr, txt); break
    fr = finish(fr, bars=(BARRAS_CINE_YOUTUBE and fmt['W'] > fmt['H']))
    fade = (fmt['L'] - g) / (1.4 * FPS)
    if fade < 1: fr = (fr * max(0.0, fade)).astype(np.uint8)                # final a negro
    if g < 8: fr = (fr * (g / 8)).astype(np.uint8)
    return fr
print("Compositor listo ✓")

In [ ]:
#@title 9 · Sonido: música de documental generada, efectos (lápiz, pincel, papel, portazo, VHS…), sonido de naturaleza y memoria corrupta
from scipy.signal import butter, sosfilt, fftconvolve
rng_a = np.random.default_rng(12)
def bp(x, lo, hi): return sosfilt(butter(2, [lo, hi], 'band', fs=SR, output='sos'), x).astype(np.float32)
def lp(x, f): return sosfilt(butter(2, f, 'low', fs=SR, output='sos'), x).astype(np.float32)
def env(n, a, r):
    e = np.ones(n, np.float32); na, nr = max(1, int(a * SR)), max(1, int(r * SR))
    e[:na] = np.linspace(0, 1, na); e[-nr:] *= np.linspace(1, 0, nr) ** 2; return e
def noise(d): return rng_a.normal(0, 1, int(d * SR)).astype(np.float32)
# --- efectos sintetizados (si pones archivos en Sonidos/SFX sonidos con estos nombres, se usan los tuyos) ---
def sfx_whoosh(d=1.0):
    n = noise(d); t = np.linspace(0, 1, len(n)); return bp(n, 300, 4000) * np.sin(np.pi * t) ** 2 * 0.5
def sfx_puerta():
    t = np.arange(int(0.9 * SR)) / SR; thump = np.sin(2 * np.pi * (60 - 25 * t) * t) * np.exp(-t * 7)
    crack = lp(noise(0.9), 2500) * np.exp(-t * 35) * 0.6; rattle = bp(noise(0.9), 800, 3000) * np.exp(-((t - 0.12) * 18) ** 2) * 0.25
    return ((thump + crack + rattle) * 0.9).astype(np.float32)
def sfx_lapiz(d=1.2):
    n = bp(noise(d), 2000, 7000); t = np.arange(len(n)) / SR; am = (0.5 + 0.5 * np.sin(2 * np.pi * 7 * t + rng_a.random() * 6)) ** 2
    return n * am * env(len(n), 0.05, 0.2) * 0.35
def sfx_pincel(d=0.45):
    n = bp(noise(d), 400, 3500); return n * np.sin(np.linspace(0, np.pi, len(n))) ** 1.5 * 0.45
def sfx_papel(d=1.0):
    n = bp(noise(d), 900, 6000); crack = (rng_a.random(len(n)) > 0.995).astype(np.float32) * rng_a.normal(0, 3, len(n)).astype(np.float32)
    return (n * 0.4 + lp(crack, 5000)) * env(len(n), 0.05, 0.3) * 0.45
def sfx_tecla():
    t = np.arange(int(0.05 * SR)) / SR; return ((bp(noise(0.05), 1500, 6000) + 0.5 * np.sin(2 * np.pi * 1900 * t)) * np.exp(-t * 120) * 0.5).astype(np.float32)
def sfx_vhs(d=1.6):
    t = np.arange(int(d * SR)) / SR; hiss = bp(noise(d), 3000, 12000) * 0.12; hum = 0.06 * np.sin(2 * np.pi * 60 * t)
    clunk = np.sin(2 * np.pi * 90 * t) * np.exp(-t * 25) * 0.7; return ((hiss + hum) * env(len(t), 0.02, 0.4) + clunk).astype(np.float32)
def sfx_camara():
    a = sfx_tecla() * 1.5; out = np.zeros(int(0.2 * SR), np.float32); out[:len(a)] += a; out[int(0.09 * SR):int(0.09 * SR) + len(a)] += a * 0.8; return out
def sfx_glitch(d=0.5):
    t = np.arange(int(d * SR)) / SR; sq = np.sign(np.sin(2 * np.pi * (300 + 900 * rng_a.random()) * t)) * 0.25
    gate = (np.floor(t * 30) % 2).astype(np.float32); return (np.round(sq * gate * 8) / 8 + bp(noise(d), 2000, 8000) * 0.1 * gate).astype(np.float32)
def sfx_boom(d=2.5):
    t = np.arange(int(d * SR)) / SR; s_ = np.sin(2 * np.pi * (48 - 18 * t / d) * t) * np.exp(-t * 1.6)
    return ((s_ + lp(noise(d), 300) * np.exp(-t * 3) * 0.3) * 0.9).astype(np.float32)
def sfx_latido():
    t = np.arange(int(0.8 * SR)) / SR; b = lambda t0: np.sin(2 * np.pi * 50 * (t - t0)) * np.exp(-np.clip(t - t0, 0, None) * 18) * (t >= t0)
    return ((b(0) + 0.7 * b(0.28)) * 0.8).astype(np.float32)
def sfx_subida(d=1.2):
    n = noise(d); t = np.linspace(0, 1, len(n)); out = np.zeros_like(n)
    for k in range(8):
        a, b_ = k * len(n) // 8, (k + 1) * len(n) // 8; out[a:b_] = lp(n[a:b_], 300 + 9000 * (k / 8) ** 2)
    return out * t ** 2 * 0.5
SFX_SYN = {'whoosh': sfx_whoosh, 'puerta': sfx_puerta, 'lapiz': sfx_lapiz, 'pincel': sfx_pincel, 'papel': sfx_papel, 'tecla': sfx_tecla,
           'vhs': sfx_vhs, 'camara': sfx_camara, 'glitch': sfx_glitch, 'boom': sfx_boom, 'latido': sfx_latido, 'subida': sfx_subida}
SFX_RE = {'whoosh': r'whoosh|swoosh|viento', 'puerta': r'puerta|door|portazo', 'lapiz': r'lapiz|lápiz|pencil|grafito', 'pincel': r'pincel|brush|pintura',
          'papel': r'papel|paper|rasg|tear|lija', 'tecla': r'tecla|maquina|máquina|typewriter', 'vhs': r'vhs|cinta|tape', 'camara': r'camara|cámara|shutter|foto',
          'glitch': r'glitch|error|estatica|estática', 'boom': r'boom|impacto|golpe|hit', 'latido': r'latido|heart|corazon|corazón', 'subida': r'subida|riser|swell'}
SND = os.path.join(PROYECTO, CARPETA_SONIDOS)
def audio_files(sub):
    d_ = os.path.join(SND, sub)
    return sorted(glob.glob(os.path.join(d_, '**', '*.*'), recursive=True)) if os.path.isdir(d_) else []
AUD_EXT = ('.wav', '.mp3', '.m4a', '.aac', '.ogg', '.flac')
SFX_FILES = [f for sub in ('SFX sonidos', 'Sonidos en crudo') for f in audio_files(sub) if f.lower().endswith(AUD_EXT)]
_sfx_cache = {}
def sfx(name, *a):
    for f in SFX_FILES:
        if re.search(SFX_RE[name], os.path.basename(f), re.I):
            if f not in _sfx_cache: _sfx_cache[f] = read_wav_mono(f)
            return _sfx_cache[f]
    return SFX_SYN[name](*a)
print("Efectos tuyos encontrados:", [os.path.basename(f) for f in SFX_FILES] or '— (uso efectos sintetizados)')

# --- música de documental generada (re menor, pads + piano con reverb + tensión) ---
def note_hz(m): return 440 * 2 ** ((m - 69) / 12)
CHORDS = [[50, 57, 62, 65], [46, 53, 58, 62], [53, 60, 65, 69], [48, 55, 60, 64]]          # Dm  Bb  F  C
def gen_music(L, segs):
    n = int(L * SR); pad = np.zeros(n, np.float32); pno = np.zeros(n, np.float32); ten = np.zeros(n, np.float32)
    tension = np.zeros(n, np.float32)
    for s in segs:
        if (s['era'] and s['era'] >= 2014) or s['transicion'] == 'memoria_corrupta' or re.search(r'odio|ataque|amenaza|cloro', s['texto'], re.I):
            tension[int(s['f0'] / FPS * SR):int(s['f1'] / FPS * SR)] = 1
    tension = np.convolve(tension, np.ones(SR) / SR, 'same').astype(np.float32)
    cd = 7.0; t_all = 0.0; ci = 0
    while t_all < L:
        a = int(t_all * SR); b = min(n, int((t_all + cd + 2) * SR)); tt = np.arange(b - a) / SR
        for m in CHORDS[ci % 4]:
            for det in (-0.12, 0.12):
                f = note_hz(m - 12) * 2 ** (det / 12)
                pad[a:b] += (np.sin(2 * np.pi * f * tt) + 0.3 * np.sin(4 * np.pi * f * tt)) * 0.03 * env(b - a, 1.8, 2.0)
        for k in range(8):                                                     # arpegio de piano, escaso
            if rng_a.random() < 0.55:
                st = int((t_all + k * cd / 8) * SR); m = CHORDS[ci % 4][k % 4] + 12 * (k % 3 == 2)
                d = min(n - st, int(2.5 * SR))
                if d <= 0: continue
                t2 = np.arange(d) / SR; f = note_hz(m)
                pno[st:st + d] += sum(np.sin(2 * np.pi * f * h * 1.0007 ** h * t2) / h ** 1.6 for h in (1, 2, 3, 4)) * np.exp(-t2 * 1.7) * 0.07
        t_all += cd; ci += 1
    tt = np.arange(n) / SR
    pulse = np.zeros(n, np.float32); lb = sfx_latido()                         # latido en la tensión
    for k in range(int(L / 0.85)):
        st = int(k * 0.85 * SR)
        if st + len(lb) < n and tension[st] > 0.5: pulse[st:st + len(lb)] += lb * 0.35
    ten = (np.sin(2 * np.pi * note_hz(81) * tt) + np.sin(2 * np.pi * note_hz(82) * tt)) * (0.5 + 0.5 * np.sin(2 * np.pi * 7 * tt)) * 0.012
    ir = (rng_a.normal(0, 1, int(2.8 * SR)) * np.exp(-np.arange(int(2.8 * SR)) / SR * 2.2)).astype(np.float32); ir /= np.abs(ir).sum() / 6
    wet = fftconvolve(pno, ir)[:n].astype(np.float32)
    mono = pad + pno * 0.7 + wet * 0.5 + (ten + pulse) * tension
    level = np.interp(tt, [0, 8, L * 0.5, L * 0.85, L], [0.55, 0.8, 0.9, 1.0, 0.8]).astype(np.float32)
    return mono * level
def load_music_folder(L):
    fs = [f for f in audio_files('Musica') if f.lower().endswith(AUD_EXT)]
    if not fs: return None
    out = np.zeros(0, np.float32)
    while len(out) < L * SR:
        for f in fs:
            x = read_wav_mono(f); xf = int(1.5 * SR)
            if len(out) > xf and len(x) > xf:
                out[-xf:] = out[-xf:] * np.linspace(1, 0, xf) + x[:xf] * np.linspace(0, 1, xf); x = x[xf:]
            out = np.concatenate([out, x])
            if len(out) >= L * SR: break
    print("Música de tu carpeta:", [os.path.basename(f) for f in fs]); return out[:int(L * SR)] * 0.6

def build_audio(fmt):
    segs, L = fmt['segs'], fmt['L'] / FPS; n = int(L * SR) + SR
    voice = np.zeros(n, np.float32); fx = np.zeros(n, np.float32); amb = np.zeros((n, 2), np.float32)
    def add(buf, x, t, g=1.0):
        a = int(t * SR); b = min(len(buf), a + len(x))
        if a < 0 or b <= a: return
        buf[a:b] += x[:b - a] * g
    def tsec(f): return f / FPS
    for k, s in enumerate(segs):
        add(voice, s['voz'], tsec(s['voz_f']))
        t0, D = tsec(s['f0']), s['D'] / FPS; tr = s['tratamiento']; era = ERA(s['era'])
        if tr == 'grafito': add(fx, sfx('lapiz', 1.6), t0 + min(0.25 * D, 2.0), 0.7)
        if tr == 'pintura':
            for q in range(5): add(fx, sfx('pincel'), t0 + q * 0.26, 0.6)
        if tr == 'periodico': add(fx, sfx('whoosh', 1.2), t0, 0.8); add(fx, sfx('papel', 0.8), t0 + 1.2, 0.7)
        if tr == 'titulo': add(fx, sfx('boom'), t0 + s['lead'] * 0.4, 0.8)
        if tr == 'anio' or (tr == 'periodico' and s['anio']):
            look = era['look']
            if look == 'vhs': add(fx, sfx('vhs'), t0 + 0.1, 0.6)
            elif look == 'digicam': add(fx, sfx('camara'), t0 + 0.15, 0.8)
            elif era['anim'] == 'glitch': add(fx, sfx('glitch'), t0 + 0.15, 0.5)
            else: add(fx, sfx('whoosh', 0.6), t0 + 0.05, 0.6)
            if era['anim'] == 'typewriter':
                for q in range(len(str(s['anio']))): add(fx, sfx('tecla'), t0 + 0.15 + q / 16, 0.7)
        for c_ in s.get('creditos', []): add(fx, sfx('pincel', 0.9), t0 + c_['a'] / FPS, 0.35)
        txt = s.get('texto_pantalla') or ''
        if era['anim'] == 'typewriter' and txt and tr not in ('titulo', 'periodico'):
            t_txt = t0 + ((s['lead'] + 1.1) if tr == 'anio' else (min(0.25 * D, 2.0) + 0.9 if tr == 'grafito' else max(0.5, s['lead'] * 0.6)))
            for q, ch in enumerate(txt):
                if ch.strip(): add(fx, sfx('tecla'), t_txt + q / 16, 0.45)
        tb = tsec(s['f1'])
        kind = s['transicion'] if k < len(segs) - 1 else 'final'
        if kind == 'desgarre': add(fx, sfx('papel', 1.1), tb - 0.55, 0.8)
        elif kind == 'enfoque_grafito': add(fx, sfx('lapiz', 1.6), tb - 0.6, 0.6)
        elif kind == 'tunel': add(fx, sfx('whoosh', 1.4), tb - 0.8, 0.9); add(fx, sfx('latido'), tb - 0.1, 0.6)
        elif kind == 'corte_accion': add(fx, sfx('puerta'), tb - 0.02, 1.0)
        elif kind == 'luz_alta': add(fx, sfx('subida', 1.0), tb - 0.95, 0.7)
        for sh in s.get('shots', []):                                     # sonido real de la naturaleza
            if sh.get('amb') is not None:
                a_ = sh['amb'] * env(len(sh['amb']), 0.3, 0.4)[:, None]
                st = int((t0 + sh['a'] / FPS) * SR); b_ = min(n, st + len(a_))
                if b_ > st: amb[st:b_] += a_[:b_ - st] * 0.16
    # música
    if MUSICA == 'sin musica': mus = np.zeros(n, np.float32)
    else:
        mus = load_music_folder(L) if MUSICA.startswith('carpeta') else None
        if mus is None: mus = gen_music(L, segs)
        mus = np.pad(mus, (0, max(0, n - len(mus))))[:n]
    gate = np.ones(n, np.float32)
    for k, s in enumerate(segs[:-1]):                                     # silencio dramático tras el portazo
        if s['transicion'] == 'corte_accion':
            a = int(tsec(s['f1']) * SR); gate[a:a + int(0.7 * SR)] = 0; gate[a + int(0.7 * SR):a + int(1.6 * SR)] = np.linspace(0, 1, int(0.9 * SR))
    rms = np.sqrt(np.convolve(voice ** 2, np.ones(2400) / 2400, 'same')); duck = 1 - 0.62 * np.clip(rms * 12, 0, 1)
    duck = np.convolve(duck, np.ones(SR // 3) / (SR // 3), 'same').astype(np.float32)
    beds = [f for f in audio_files('Sonidos remasterizados (ambiente)') if f.lower().endswith(AUD_EXT)]
    if beds:
        bed = np.concatenate([read_wav_mono(f) for f in beds]); bed = np.tile(bed, int(n / max(1, len(bed))) + 1)[:n]; amb += (bed * 0.12)[:, None]
    mix = np.stack([voice, voice], 1) * 1.0 + (mus * gate * duck)[:, None] * 0.85 + fx[:, None] * 0.8 + amb * duck[:, None]
    # memoria corrupta: la imagen se congela y el audio sigue 2-3 s distorsionado
    for s in segs[:-1]:
        if s['transicion'] != 'memoria_corrupta': continue
        a = int(tsec(s['f1']) * SR); g_ = s['gap'] / FPS; m = int(g_ * SR); src = mix[max(0, a - int(0.9 * SR)):a].copy()
        out = np.zeros((m, 2), np.float32); pos = 0; chunk = int(0.3 * SR)
        while pos < m:
            piece = src[-chunk:] if len(src) >= chunk else src
            sp = 1 - 0.5 * pos / m; idx = (np.arange(int(len(piece) / sp)) * sp).astype(int); piece = piece[idx[idx < len(piece)]]
            out[pos:pos + len(piece)] = piece[:m - pos]; pos += len(piece); chunk = max(int(0.05 * SR), int(chunk * 0.8))
        out = np.round(out * 12) / 12; out[::1] = np.repeat(out[::6], 6, 0)[:m]
        out *= np.linspace(1, 0.2, m)[:, None]; out[:, 0] += np.pad(sfx('glitch', 0.6), (0, m))[:m] * 0.4; out[:, 1] += np.pad(sfx('glitch', 0.6), (int(1.0 * SR), m))[:m] * 0.4
        mix[a:a + m] = out; f_in = min(int(0.6 * SR), n - a - m); mix[a + m:a + m + f_in] *= np.linspace(0, 1, f_in)[:, None]
    mix = mix[:int(L * SR)]; mix /= max(1.0, np.abs(mix).max() / 0.95)
    raw = f"{WORK}/audio/{fmt['name']}_mix_raw.wav"; out = f"{WORK}/audio/{fmt['name']}_mix.wav"
    import soundfile as sf
    sf.write(raw, mix, SR, subtype='FLOAT')
    run(['ffmpeg', '-y', '-v', 'error', '-i', raw, '-af', 'loudnorm=I=-14:TP=-1.5:LRA=11', '-ar', str(SR), out])
    fmt['audio'] = out; print(f"🔊 Audio {fmt['name']} listo ({L:.0f} s)")
for fmt in FORMATOS: build_audio(fmt)

In [ ]:
#@title 10 · Vista previa: hoja de fotogramas de cada segmento (se guarda en tu Drive antes del render)
for fmt in FORMATOS:
    fmt['subs_list'] = make_subs(fmt)
    picks = []
    for s in fmt['segs']: picks += [s['f0'] + int(s['D'] * 0.3), s['f0'] + int(s['D'] * 0.75)]
    for s in fmt['segs'][:-1]:                                    # un cuadro en medio de cada transición
        picks.append(s['f1'] + (s['gap'] // 2 if s['gap'] else (s['post'] - s['pre']) // 2))
    picks = sorted(set(min(fmt['L'] - 1, max(0, p_)) for p_ in picks)); got = {}
    render_frames(fmt, lambda g, fr: got.__setitem__(g, post(fmt, g, fr.copy(), fmt['subs_list'])), only=set(picks))
    tw = 480 if fmt['W'] > fmt['H'] else 270; th = int(tw * fmt['H'] / fmt['W']); cols = 6
    ims = [cv2.resize(got[p_], (tw, th)) for p_ in picks if p_ in got]
    for im, p_ in zip(ims, picks): cv2.putText(im, f"{p_ / FPS:.1f}s", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    rows = [np.hstack(ims[i:i + cols] + [np.zeros_like(ims[0])] * (cols - len(ims[i:i + cols]))) for i in range(0, len(ims), cols)]
    sheet = np.vstack(rows); path = os.path.join(OUT_DIR, f"vista_previa_{fmt['name']}.jpg")
    cv2.imwrite(path, sheet, [cv2.IMWRITE_JPEG_QUALITY, 85]); print("Vista previa:", path)
    try:
        from IPython.display import Image as IPImage, display; display(IPImage(path, width=1200))
    except Exception: pass

In [ ]:
#@title 11 · Render final: YouTube 16:9 (1920×1080) y TikTok 9:16 (1080×1920)
SALIDAS = []
for fmt in FORMATOS:
    out = os.path.join(WORK, f"documental_furry_intro_{fmt['name']}.mp4"); subs = fmt.get('subs_list') or make_subs(fmt)
    cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f"{fmt['W']}x{fmt['H']}", '-r', str(FPS), '-i', '-',
           '-i', fmt['audio'], '-map', '0:v', '-map', '1:a', *VENC, '-pix_fmt', 'yuv420p', '-color_primaries', 'bt709', '-color_trc', 'bt709',
           '-colorspace', 'bt709', '-c:a', 'aac', '-b:a', '256k', '-t', f"{fmt['L'] / FPS:.3f}", '-movflags', '+faststart', out]
    pipe = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE); t0 = time.time()
    def emit(g, fr):
        try: pipe.stdin.write(post(fmt, g, fr, subs).tobytes())
        except BrokenPipeError: raise RuntimeError("ffmpeg se detuvo: " + pipe.stderr.read().decode()[-2000:])
        if g % 48 == 0: print(f"\r  {fmt['name']}: {g}/{fmt['L']} frames ({(g + 1) / max(1e-6, time.time() - t0):.1f} fps)", end='')
    n = render_frames(fmt, emit)
    pipe.stdin.close(); err = pipe.stderr.read().decode(); pipe.wait()
    if pipe.returncode != 0: raise RuntimeError(err[-2000:])
    print(f"\n✓ {out} ({n / FPS:.0f} s, {(time.time() - t0) / 60:.1f} min)"); SALIDAS.append(out)

In [ ]:
#@title 12 · Guardar en tu Drive + notas del editor (tomas usadas, guion, créditos)
notas = [f"# {TITULO_DOC} — intro (editor IA)\n", f"Guion: {GUION_FUENTE} · Voz: {', '.join(sorted(VOZ_USADA)) or '—'} · Música: {MUSICA}\n"]
for fmt in FORMATOS:
    notas.append(f"\n## {fmt['name']} ({fmt['L'] / FPS:.0f} s)\n")
    for i, s in enumerate(fmt['segs']):
        tomas = ', '.join(f"{clips[sh['clip']]['name']} @{sh['t0'] + PAD * sh['speed']:.1f}s" for sh in s['shots'])
        notas.append(f"{i + 1}. [{s['f0'] / FPS:6.1f}s] {s['tratamiento']}{' ' + str(s['anio']) if s['anio'] else ''} → {s['transicion']}\n"
                     f"   Voz: {s['texto']}\n   Tomas: {tomas}\n")
notas.append("\nNota: las páginas de periódico son RECREACIONES hechas para el documental (no son portadas reales).\n")
with open(os.path.join(OUT_DIR, 'notas_del_editor_documental.md'), 'w', encoding='utf-8') as f_: f_.writelines(notas)
for p_ in SALIDAS:
    shutil.copy(p_, OUT_DIR); print("✓ En tu Drive:", os.path.join(OUT_DIR, os.path.basename(p_)))
print("✓ Notas:", os.path.join(OUT_DIR, 'notas_del_editor_documental.md'))